# DigiHero: Stratified analyses

This notebook produces stratified descriptive comparisons for key outcomes (AI use, health-related AI use, side effects, and downstream behaviors) across participant subgroups (e.g., age bands, gender, daily functioning, diagnosis, treatment) in the DigiHero sample.


## Analysis sample filtering

This section applies the manuscript-aligned inclusion criteria (e.g., non-missing age and gender and a valid AI-use response where required) to define the analytic sample used across stratified summaries.


In [1]:
import pandas as pd
import numpy as np

# --- Dataset path and core variables ---
DATA_PATH = r"C:\\Dokumente\\Submissions\\LLM_Survey\\Daten\\KIDaten_renamed.csv"
ki_col = "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
gender_col = "gender_filled"
birth_year_col = "birth_year_filled"
REFERENCE_YEAR = 2026

# Robust CSV read (tries common separators)
def read_table(path):
    path_l = str(path).lower()
    if path_l.endswith('.csv'):
        attempts = []
        trials = [(',', 'c'), (';', 'c'), ('	', 'c'), ('|', 'c'), (',', 'python'), (';', 'python')]
        for sep, eng in trials:
            try:
                dfx = pd.read_csv(path, sep=sep, engine=eng, low_memory=False)
                if dfx.shape[1] > 1:
                    print(f"Loaded CSV with sep={repr(sep)}, engine={eng}")
                    return dfx
                attempts.append(f"sep={repr(sep)} engine={eng}: only 1 column")
            except Exception as e:
                attempts.append(f"sep={repr(sep)} engine={eng}: {type(e).__name__}")
        raise ValueError("Could not parse CSV. Attempts: " + " | ".join(attempts))
    return pd.read_excel(path)

df = read_table(DATA_PATH)
print(f"Rows: {len(df):,}, Columns: {df.shape[1]:,}")
print(f"Loaded dataset: {DATA_PATH}")
print(f"Rows: {len(df):,} | Columns: {df.shape[1]:,}")

required_cols = [ki_col, gender_col, birth_year_col]
missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

Loaded CSV with sep=';', engine=c
Rows: 29,571, Columns: 577
Loaded dataset: C:\\Dokumente\\Submissions\\LLM_Survey\\Daten\\KIDaten_renamed.csv
Rows: 29,571 | Columns: 577


In [2]:
# --- Analysis sample filtering (aligned with regression notebooks) ---
# Inclusion criteria:
# - valid Yes/No response on the AI-use item
# - non-missing gender and birth year (used to derive age)
yes_no = ["Ja", "ja", "Yes", "yes", "Nein", "nein", "No", "no"]

# Valid AI-use response (after stripping whitespace)
ai_clean = df[ki_col].astype(str).str.strip()
has_valid_ai = ai_clean.isin(yes_no)

# Required demographics present
has_gender = df[gender_col].notna()
has_age = df[birth_year_col].notna()
has_demo = has_gender & has_age

# Final kept sample
mask_keep = has_valid_ai & has_demo
df_base = df.loc[mask_keep].copy()

# Age used for reporting
df_base["age_2026"] = REFERENCE_YEAR - pd.to_numeric(df_base[birth_year_col], errors="coerce")
df_base = df_base.dropna(subset=["age_2026"])

n_total = len(df)
n_invalid_ai = int((~has_valid_ai).sum())
n_missing_gender = int((~has_gender).sum())
n_missing_age = int((~has_age).sum())
n_kept = len(df_base)
n_excluded_total = n_total - n_kept

print("Exclusions (reasons can overlap):")
print(f"  Missing gender:              {n_missing_gender:>6}")
print(f"  Missing age:                 {n_missing_age:>6}")
print(f"  Non-Yes/No on KI question:   {n_invalid_ai:>6}")
print(f"  Total excluded:              {n_excluded_total:>6}")
print(f"  Analysis sample (kept):      {n_kept:>6}")

Exclusions (reasons can overlap):
  Missing gender:                 100
  Missing age:                    127
  Non-Yes/No on KI question:      526
  Total excluded:                 661
  Analysis sample (kept):       28910


## Age band construction

We construct age bands for stratified analyses. 


In [3]:
# --- Age band distribution (descriptives) ---
# Bins are left-closed/right-open: [18, 28), [28, 38),  [78, inf)
bins = [18, 28, 38, 48, 58, 68, 78, np.inf]
labels = ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]

df_age = df_base.loc[df_base["age_2026"] >= 18].copy()
df_age["age_band"] = pd.cut(
    df_age["age_2026"],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True,
)

age_table = (
    df_age["age_band"]
    .value_counts(dropna=False)
    .reindex(labels)
    .fillna(0)
    .astype(int)
    .rename("n")
    .to_frame()
)
age_table["pct"] = (age_table["n"] / age_table["n"].sum() * 100).round(1)

print(f"Rows with age >= 18 in kept sample: {len(df_age):,}")
print("\nAge-band distribution:")
print(age_table)

Rows with age >= 18 in kept sample: 28,910

Age-band distribution:
             n   pct
age_band            
18-27      570   2.0
28-37     2213   7.7
38-47     3886  13.4
48-57     5149  17.8
58-67     8554  29.6
68-77     6799  23.5
78+       1739   6.0


## Stratified outcomes: AI use, general health-related AI use and mental health-related AI use

### Stratification by age band


In [4]:
import pandas as pd
import numpy as np

# =========================================================
# Assumes you already have df_base from your previous cells:
# - valid AI Yes/No only
# - non-missing gender
# - non-missing age_2026
# =========================================================

# ---------- Age bands ----------
bins = [18, 28, 38, 48, 58, 68, 78, np.inf]
labels = ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]

df_age = df_base.loc[df_base["age_2026"] >= 18].copy()
df_age["age_band"] = pd.cut(
    df_age["age_2026"],
    bins=bins,
    labels=labels,
    right=False,         # [18,28), [28,38), ...
    include_lowest=True,
)

# ---------- Helper to build count/% table ----------
def build_binary_age_table(data, outcome_col, user_label, non_user_label):
    """
    outcome_col must be binary: 1=user, 0=non-user, NaN=missing
    """
    d = data.dropna(subset=[outcome_col]).copy()

    counts = (
        d.groupby(["age_band", outcome_col], observed=False)
         .size()
         .unstack(fill_value=0)
         .reindex(labels, fill_value=0)
    )

    # Ensure both 0 and 1 columns exist
    if 0 not in counts.columns:
        counts[0] = 0
    if 1 not in counts.columns:
        counts[1] = 0
    counts = counts[[1, 0]]

    out = counts.rename(columns={1: f"n_{user_label}", 0: f"n_{non_user_label}"})
    out["n_total"] = out[f"n_{user_label}"] + out[f"n_{non_user_label}"]
    out[f"pct_{user_label}"] = (out[f"n_{user_label}"] / out["n_total"] * 100).round(1)
    out[f"pct_{non_user_label}"] = (out[f"n_{non_user_label}"] / out["n_total"] * 100).round(1)
    return out

# =========================================================
# A) AI use (whole filtered sample)
# =========================================================
# ki_col should already be defined in your notebook
ai_s = df_age[ki_col].astype(str).str.strip().str.lower()
df_age["ai_use_bin"] = np.where(
    ai_s.isin(["ja", "yes"]), 1,
    np.where(ai_s.isin(["nein", "no"]), 0, np.nan)
)

ai_table = build_binary_age_table(
    data=df_age,
    outcome_col="ai_use_bin",
    user_label="ai_user",
    non_user_label="non_ai_user"
)

print("\nAI use by age band (whole filtered sample):")
print(ai_table)

# =========================================================
# B) Medical AI use (among AI users only), same coding as regression
# =========================================================
purpose_cols_13 = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in df_age.columns]

non_medical_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_med_set = {x.strip().lower() for x in non_medical_answers}

def norm_str(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() == "nan":
        return ""
    return v.lower()

# Restrict to AI users first (as in your regression)
df_ai_users = df_age.loc[df_age["ai_use_bin"] == 1].copy()

if purpose_cols_13:
    med_vals = []
    for _, row in df_ai_users[purpose_cols_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_13]
        answered = [v for v in vals if v != ""]
        if len(answered) == 0:
            med_vals.append(np.nan)              # no answer in all 3 items
        elif any(v not in non_med_set for v in answered):
            med_vals.append(1)                   # substantive medical use
        else:
            med_vals.append(0)                   # only non-medical answers
    df_ai_users["med_use"] = med_vals

    med_table = build_binary_age_table(
        data=df_ai_users,
        outcome_col="med_use",
        user_label="med_user",
        non_user_label="med_non_user"
    )

    print("\nMedical AI use by age band (among AI users only):")
    print(med_table)
else:
    print("\nMedical AI use table skipped: none of purpose_cols_13 found in dataset.")

# =========================================================
# C) Emotional AI use (among AI users only), same coding as regression
# =========================================================
purpose_cols_4_13 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in df_ai_users.columns]

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_use_set = {x.strip().lower() for x in non_use_answers}

if purpose_cols_4_13:
    emo_vals = []
    for _, row in df_ai_users[purpose_cols_4_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_4_13]
        answered = [v for v in vals if v != ""]
        if len(answered) == 0:
            emo_vals.append(np.nan)              # no answer in all 9 items
        elif any(v not in non_use_set for v in answered):
            emo_vals.append(1)                   # substantive emotional use
        else:
            emo_vals.append(0)                   # only non-use answers
    df_ai_users["emo_use"] = emo_vals

    emo_table = build_binary_age_table(
        data=df_ai_users,
        outcome_col="emo_use",
        user_label="emo_user",
        non_user_label="emo_non_user"
    )

    print("\nEmotional AI use by age band (among AI users only):")
    print(emo_table)
else:
    print("\nEmotional AI use table skipped: none of purpose_cols_4_13 found in dataset.")


AI use by age band (whole filtered sample):
ai_use_bin  n_ai_user  n_non_ai_user  n_total  pct_ai_user  pct_non_ai_user
age_band                                                                   
18-27             467            103      570         81.9             18.1
28-37            1769            444     2213         79.9             20.1
38-47            2776           1110     3886         71.4             28.6
48-57            3173           1976     5149         61.6             38.4
58-67            4095           4459     8554         47.9             52.1
68-77            2296           4503     6799         33.8             66.2
78+               482           1257     1739         27.7             72.3

Medical AI use by age band (among AI users only):
med_use   n_med_user  n_med_non_user  n_total  pct_med_user  pct_med_non_user
age_band                                                                     
18-27            312             155      467          66.8     

### Stratification by gender


In [5]:
import pandas as pd
import numpy as np

# =========================================================
# FULL BLOCK: AI / medical AI / emotional AI by gender
# Includes female, male, other
#
# Assumes:
# - df_base exists (your filtered base sample)
# - ki_col and gender_col are defined
# - df_base contains age_2026 if you want to additionally filter on age >= 18
# =========================================================

d = df_base.copy()

# Optional: keep adults only (uncomment if desired)
# d = d.loc[d["age_2026"] >= 18].copy()

# ---------- 1) AI use binary ----------
ai_s = d[ki_col].astype(str).str.strip().str.lower()
d["ai_use_bin"] = np.where(
    ai_s.isin(["ja", "yes"]), 1,
    np.where(ai_s.isin(["nein", "no"]), 0, np.nan)
)

# ---------- 2) Gender mapping: female / male / other ----------
g = d[gender_col].astype(str).str.strip().str.lower()

is_male = g.isin(["m", "male", "männlich", "maennlich"])
is_female = g.isin(["f", "female", "weiblich"])
is_other = g.isin([
    "other", "divers", "diverse", "non-binary", "nonbinary", "nb", "anders"
])

# Robust approach:
# - explicit female/male/other
# - any non-missing value not recognized as female/male => other
d["gender_group"] = pd.Series(pd.NA, index=d.index, dtype="string")
d.loc[is_female, "gender_group"] = "female"
d.loc[is_male, "gender_group"] = "male"
d.loc[is_other, "gender_group"] = "other"

unknown_nonmissing = d[gender_col].notna() & d["gender_group"].isna()
d.loc[unknown_nonmissing, "gender_group"] = "other"

gender_order = ["female", "male", "other"]
d_gender = d[d["gender_group"].notna()].copy()

# ---------- 3) Helper to build count/% tables by gender ----------
def build_binary_gender_table(data, outcome_col, user_label, non_user_label):
    tmp = data.dropna(subset=[outcome_col]).copy()

    counts = (
        tmp.groupby(["gender_group", outcome_col], observed=False)
           .size()
           .unstack(fill_value=0)
           .reindex(gender_order, fill_value=0)
    )

    # Ensure both classes exist
    if 0 not in counts.columns:
        counts[0] = 0
    if 1 not in counts.columns:
        counts[1] = 0
    counts = counts[[1, 0]]

    out = counts.rename(columns={1: f"n_{user_label}", 0: f"n_{non_user_label}"})
    out["n_total"] = out[f"n_{user_label}"] + out[f"n_{non_user_label}"]
    out[f"pct_{user_label}"] = (out[f"n_{user_label}"] / out["n_total"] * 100).round(1)
    out[f"pct_{non_user_label}"] = (out[f"n_{non_user_label}"] / out["n_total"] * 100).round(1)
    return out

# =========================================================
# A) AI use by gender (female/male/other)
# =========================================================
ai_gender_table = build_binary_gender_table(
    data=d_gender,
    outcome_col="ai_use_bin",
    user_label="ai_user",
    non_user_label="non_ai_user"
)

print("\nAI use by gender (female/male/other):")
print(ai_gender_table)

# =========================================================
# B) Medical AI use by gender (among AI users only)
# Same coding logic as in your regression notebook
# =========================================================
purpose_cols_13 = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in d_gender.columns]

non_medical_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_med_set = {x.strip().lower() for x in non_medical_answers}

def norm_str(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() == "nan":
        return ""
    return v.lower()

d_ai = d_gender.loc[d_gender["ai_use_bin"] == 1].copy()

if purpose_cols_13:
    med_vals = []
    for _, row in d_ai[purpose_cols_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            med_vals.append(np.nan)   # no answer in all 3 items
        elif any(v not in non_med_set for v in answered):
            med_vals.append(1)        # substantive medical use
        else:
            med_vals.append(0)        # only non-medical answers

    d_ai["med_use"] = med_vals

    med_gender_table = build_binary_gender_table(
        data=d_ai,
        outcome_col="med_use",
        user_label="med_user",
        non_user_label="med_non_user"
    )

    print("\nMedical AI use by gender (among AI users only):")
    print(med_gender_table)
else:
    print("\nMedical AI use table skipped: required medical-use columns not found.")

# =========================================================
# C) Emotional AI use by gender (among AI users only)
# Same coding logic as in your regression notebook
# =========================================================
purpose_cols_4_13 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in d_ai.columns]

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_use_set = {x.strip().lower() for x in non_use_answers}

if purpose_cols_4_13:
    emo_vals = []
    for _, row in d_ai[purpose_cols_4_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_4_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            emo_vals.append(np.nan)   # no answer in all 9 items
        elif any(v not in non_use_set for v in answered):
            emo_vals.append(1)        # substantive emotional use
        else:
            emo_vals.append(0)        # only non-use answers

    d_ai["emo_use"] = emo_vals

    emo_gender_table = build_binary_gender_table(
        data=d_ai,
        outcome_col="emo_use",
        user_label="emo_user",
        non_user_label="emo_non_user"
    )

    print("\nEmotional AI use by gender (among AI users only):")
    print(emo_gender_table)
else:
    print("\nEmotional AI use table skipped: required emotional-use columns not found.")


AI use by gender (female/male/other):
ai_use_bin    n_ai_user  n_non_ai_user  n_total  pct_ai_user  pct_non_ai_user
gender_group                                                                 
female             9000           8103    17103         52.6             47.4
male               6017           5716    11733         51.3             48.7
other                41             33       74         55.4             44.6

Medical AI use by gender (among AI users only):
med_use       n_med_user  n_med_non_user  n_total  pct_med_user  \
gender_group                                                      
female              5538            3455     8993          61.6   
male                3168            2844     6012          52.7   
other                 21              20       41          51.2   

med_use       pct_med_non_user  
gender_group                    
female                    38.4  
male                      47.3  
other                     48.8  

Emotional AI use by 

### Stratification by daily functioning


In [6]:
import pandas as pd
import numpy as np

# =========================================================
# FULL BLOCK: AI / medical AI / emotional AI by EQ5D5L_3
#
# EQ5D5L_3 coding:
#   a=1, b=2, c=3, d=4, e=5, else missing
#
# Assumes:
# - df_base exists
# - ki_col is defined (AI use question column)
# =========================================================

d = df_base.copy()

# ---------- 1) AI use binary ----------
ai_s = d[ki_col].astype(str).str.strip().str.lower()
d["ai_use_bin"] = np.where(
    ai_s.isin(["ja", "yes"]), 1,
    np.where(ai_s.isin(["nein", "no"]), 0, np.nan)
)

# ---------- 2) EQ5D5L_3 -> eq5d_group ----------
eq_raw = d["EQ5D5L_3"].astype(str).str.strip().str.lower()
eq_map = {"a": 1, "b": 2, "c": 3, "d": 4, "e": 5}
d["eq5d_daily"] = eq_raw.map(eq_map)

d["eq5d_daily"] = pd.to_numeric(d["eq5d_daily"], errors="coerce")
d["eq5d_daily"] = d["eq5d_daily"].where(d["eq5d_daily"].isin([1, 2, 3, 4, 5]), np.nan)

eq_labels = {
    1: "1 (no problems)",
    2: "2 (slight)",
    3: "3 (moderate)",
    4: "4 (severe)",
    5: "5 (extreme)",
}
d["eq5d_group"] = d["eq5d_daily"].map(eq_labels)

eq_order = [eq_labels[i] for i in [1, 2, 3, 4, 5]]
d_eq = d[d["eq5d_group"].notna()].copy()

# ---------- 3) Helper ----------
def build_binary_eq_table(data, outcome_col, user_label, non_user_label):
    tmp = data.dropna(subset=[outcome_col]).copy()

    counts = (
        tmp.groupby(["eq5d_group", outcome_col], observed=False)
           .size()
           .unstack(fill_value=0)
           .reindex(eq_order, fill_value=0)
    )

    if 0 not in counts.columns:
        counts[0] = 0
    if 1 not in counts.columns:
        counts[1] = 0
    counts = counts[[1, 0]]

    out = counts.rename(columns={1: f"n_{user_label}", 0: f"n_{non_user_label}"})
    out["n_total"] = out[f"n_{user_label}"] + out[f"n_{non_user_label}"]
    out[f"pct_{user_label}"] = (out[f"n_{user_label}"] / out["n_total"] * 100).round(1)
    out[f"pct_{non_user_label}"] = (out[f"n_{non_user_label}"] / out["n_total"] * 100).round(1)

    n_included = len(tmp)
    n_by_eq = tmp["eq5d_group"].value_counts().reindex(eq_order, fill_value=0).astype(int)
    return out, n_included, n_by_eq

def norm_str(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() == "nan":
        return ""
    return v.lower()

# =========================================================
# A) AI use by EQ5D5L_3
# =========================================================
ai_eq_table, n_ai_included, n_ai_by_eq = build_binary_eq_table(
    data=d_eq,
    outcome_col="ai_use_bin",
    user_label="ai_user",
    non_user_label="non_ai_user"
)

print("\nAI use by EQ5D5L_3:")
print(f"Included N for AI outcome: {n_ai_included:,}")
print("Included N by EQ5D group:")
print(n_ai_by_eq)
print(ai_eq_table)

# =========================================================
# B) Medical AI use by EQ5D5L_3 (among AI users only)
# =========================================================
purpose_cols_13 = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in d_eq.columns]

non_medical_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_med_set = {x.strip().lower() for x in non_medical_answers}

d_ai = d_eq.loc[d_eq["ai_use_bin"] == 1].copy()

if purpose_cols_13:
    med_vals = []
    for _, row in d_ai[purpose_cols_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            med_vals.append(np.nan)
        elif any(v not in non_med_set for v in answered):
            med_vals.append(1)
        else:
            med_vals.append(0)

    d_ai["med_use"] = med_vals

    med_eq_table, n_med_included, n_med_by_eq = build_binary_eq_table(
        data=d_ai,
        outcome_col="med_use",
        user_label="med_user",
        non_user_label="med_non_user"
    )

    print("\nMedical AI use by EQ5D5L_3 (among AI users only):")
    print(f"Included N for medical outcome: {n_med_included:,}")
    print("Included N by EQ5D group:")
    print(n_med_by_eq)
    print(med_eq_table)
else:
    print("\nMedical AI use table skipped: required medical-use columns not found.")

# =========================================================
# C) Emotional AI use by EQ5D5L_3 (among AI users only)
# =========================================================
purpose_cols_4_13 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in d_ai.columns]

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_use_set = {x.strip().lower() for x in non_use_answers}

if purpose_cols_4_13:
    emo_vals = []
    for _, row in d_ai[purpose_cols_4_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_4_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            emo_vals.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            emo_vals.append(1)
        else:
            emo_vals.append(0)

    d_ai["emo_use"] = emo_vals

    emo_eq_table, n_emo_included, n_emo_by_eq = build_binary_eq_table(
        data=d_ai,
        outcome_col="emo_use",
        user_label="emo_user",
        non_user_label="emo_non_user"
    )

    print("\nEmotional AI use by EQ5D5L_3 (among AI users only):")
    print(f"Included N for emotional outcome: {n_emo_included:,}")
    print("Included N by EQ5D group:")
    print(n_emo_by_eq)
    print(emo_eq_table)
else:
    print("\nEmotional AI use table skipped: required emotional-use columns not found.")


AI use by EQ5D5L_3:
Included N for AI outcome: 12,975
Included N by EQ5D group:
eq5d_group
1 (no problems)    6583
2 (slight)         3910
3 (moderate)       1815
4 (severe)          592
5 (extreme)          75
Name: count, dtype: int64
ai_use_bin       n_ai_user  n_non_ai_user  n_total  pct_ai_user  \
eq5d_group                                                        
1 (no problems)       3152           3431     6583         47.9   
2 (slight)            1795           2115     3910         45.9   
3 (moderate)           735           1080     1815         40.5   
4 (severe)             239            353      592         40.4   
5 (extreme)             39             36       75         52.0   

ai_use_bin       pct_non_ai_user  
eq5d_group                        
1 (no problems)             52.1  
2 (slight)                  54.1  
3 (moderate)                59.5  
4 (severe)                  59.6  
5 (extreme)                 48.0  

Medical AI use by EQ5D5L_3 (among AI users onl

### Stratification by mental health diagnosis status


In [7]:
import pandas as pd
import numpy as np

# =========================================================
# FULL BLOCK: AI / medical AI / emotional AI by diagnosis
# Diagnosis coding follows your regression logic:
#   Diagnose_selbst_dich: a = yes, b = no, else missing
#
# Assumes:
# - df_base exists (your filtered base sample)
# - ki_col is defined
# - column Diagnose_selbst_dich exists
# =========================================================

d = df_base.copy()

diag_raw_col = "Diagnose_selbst_dich"
if diag_raw_col not in d.columns:
    raise KeyError(f"Missing required column: {diag_raw_col}")

# ---------- 1) AI use binary ----------
ai_s = d[ki_col].astype(str).str.strip().str.lower()
d["ai_use_bin"] = np.where(
    ai_s.isin(["ja", "yes"]), 1,
    np.where(ai_s.isin(["nein", "no"]), 0, np.nan)
)

# ---------- 2) Diagnosis mapping (yes/no) ----------
ds = d[diag_raw_col].astype(str).str.strip().str.lower()
d["diagnosis_bin"] = np.where(ds == "a", 1, np.where(ds == "b", 0, np.nan))
d["diagnosis_group"] = np.where(d["diagnosis_bin"] == 1, "yes",
                         np.where(d["diagnosis_bin"] == 0, "no", pd.NA))

diag_order = ["no", "yes"]
d_diag = d[d["diagnosis_group"].notna()].copy()

# ---------- 3) Helper ----------
def build_binary_diag_table(data, outcome_col, user_label, non_user_label):
    tmp = data.dropna(subset=[outcome_col]).copy()

    counts = (
        tmp.groupby(["diagnosis_group", outcome_col], observed=False)
           .size()
           .unstack(fill_value=0)
           .reindex(diag_order, fill_value=0)
    )

    if 0 not in counts.columns:
        counts[0] = 0
    if 1 not in counts.columns:
        counts[1] = 0
    counts = counts[[1, 0]]

    out = counts.rename(columns={1: f"n_{user_label}", 0: f"n_{non_user_label}"})
    out["n_total"] = out[f"n_{user_label}"] + out[f"n_{non_user_label}"]
    out[f"pct_{user_label}"] = (out[f"n_{user_label}"] / out["n_total"] * 100).round(1)
    out[f"pct_{non_user_label}"] = (out[f"n_{non_user_label}"] / out["n_total"] * 100).round(1)
    return out, len(tmp)

# =========================================================
# A) AI use by diagnosis (yes/no)
# =========================================================
ai_diag_table, n_ai_included = build_binary_diag_table(
    data=d_diag,
    outcome_col="ai_use_bin",
    user_label="ai_user",
    non_user_label="non_ai_user"
)

print("\nAI use by diagnosis (yes/no):")
print(f"Included N for AI outcome: {n_ai_included:,}")
print(ai_diag_table)

# =========================================================
# B) Medical AI use by diagnosis (among AI users only)
# Same coding logic as your regression notebook
# =========================================================
purpose_cols_13 = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in d_diag.columns]

non_medical_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_med_set = {x.strip().lower() for x in non_medical_answers}

def norm_str(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() == "nan":
        return ""
    return v.lower()

d_ai = d_diag.loc[d_diag["ai_use_bin"] == 1].copy()

if purpose_cols_13:
    med_vals = []
    for _, row in d_ai[purpose_cols_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            med_vals.append(np.nan)
        elif any(v not in non_med_set for v in answered):
            med_vals.append(1)
        else:
            med_vals.append(0)

    d_ai["med_use"] = med_vals

    med_diag_table, n_med_included = build_binary_diag_table(
        data=d_ai,
        outcome_col="med_use",
        user_label="med_user",
        non_user_label="med_non_user"
    )

    print("\nMedical AI use by diagnosis (among AI users only):")
    print(f"Included N for medical outcome: {n_med_included:,}")
    print(med_diag_table)
else:
    print("\nMedical AI use table skipped: required medical-use columns not found.")

# =========================================================
# C) Emotional AI use by diagnosis (among AI users only)
# Same coding logic as your regression notebook
# =========================================================
purpose_cols_4_13 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in d_ai.columns]

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_use_set = {x.strip().lower() for x in non_use_answers}

if purpose_cols_4_13:
    emo_vals = []
    for _, row in d_ai[purpose_cols_4_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_4_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            emo_vals.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            emo_vals.append(1)
        else:
            emo_vals.append(0)

    d_ai["emo_use"] = emo_vals

    emo_diag_table, n_emo_included = build_binary_diag_table(
        data=d_ai,
        outcome_col="emo_use",
        user_label="emo_user",
        non_user_label="emo_non_user"
    )

    print("\nEmotional AI use by diagnosis (among AI users only):")
    print(f"Included N for emotional outcome: {n_emo_included:,}")
    print(emo_diag_table)
else:
    print("\nEmotional AI use table skipped: required emotional-use columns not found.")


AI use by diagnosis (yes/no):
Included N for AI outcome: 12,300
ai_use_bin       n_ai_user  n_non_ai_user  n_total  pct_ai_user  \
diagnosis_group                                                   
no                    4627           5576    10203         45.3   
yes                   1043           1054     2097         49.7   

ai_use_bin       pct_non_ai_user  
diagnosis_group                   
no                          54.7  
yes                         50.3  

Medical AI use by diagnosis (among AI users only):
Included N for medical outcome: 5,666
med_use          n_med_user  n_med_non_user  n_total  pct_med_user  \
diagnosis_group                                                      
no                     2493            2130     4623          53.9   
yes                     654             389     1043          62.7   

med_use          pct_med_non_user  
diagnosis_group                    
no                           46.1  
yes                          37.3  

Emotional 

### Stratification by psychological treatment status


In [8]:
import pandas as pd
import numpy as np

# =========================================================
# FULL BLOCK: AI / medical AI / emotional AI by treatment
# Treatment coding follows your regression logic:
#   Behandlung_dich: a = yes, b = no, else missing
#
# Assumes:
# - df_base exists (your filtered base sample)
# - ki_col is defined
# - column Behandlung_dich exists
# =========================================================

d = df_base.copy()

treat_raw_col = "Behandlung_dich"
if treat_raw_col not in d.columns:
    raise KeyError(f"Missing required column: {treat_raw_col}")

# ---------- 1) AI use binary ----------
ai_s = d[ki_col].astype(str).str.strip().str.lower()
d["ai_use_bin"] = np.where(
    ai_s.isin(["ja", "yes"]), 1,
    np.where(ai_s.isin(["nein", "no"]), 0, np.nan)
)

# ---------- 2) Treatment mapping (yes/no) ----------
bh = d[treat_raw_col].astype(str).str.strip().str.lower()
d["treatment_bin"] = np.where(bh == "a", 1, np.where(bh == "b", 0, np.nan))
d["treatment_group"] = np.where(
    d["treatment_bin"] == 1, "yes",
    np.where(d["treatment_bin"] == 0, "no", pd.NA)
)

treat_order = ["no", "yes"]
d_treat = d[d["treatment_group"].notna()].copy()

# ---------- 3) Helper ----------
def build_binary_treat_table(data, outcome_col, user_label, non_user_label):
    tmp = data.dropna(subset=[outcome_col]).copy()

    counts = (
        tmp.groupby(["treatment_group", outcome_col], observed=False)
           .size()
           .unstack(fill_value=0)
           .reindex(treat_order, fill_value=0)
    )

    if 0 not in counts.columns:
        counts[0] = 0
    if 1 not in counts.columns:
        counts[1] = 0
    counts = counts[[1, 0]]

    out = counts.rename(columns={1: f"n_{user_label}", 0: f"n_{non_user_label}"})
    out["n_total"] = out[f"n_{user_label}"] + out[f"n_{non_user_label}"]
    out[f"pct_{user_label}"] = (out[f"n_{user_label}"] / out["n_total"] * 100).round(1)
    out[f"pct_{non_user_label}"] = (out[f"n_{non_user_label}"] / out["n_total"] * 100).round(1)
    return out, len(tmp)

# =========================================================
# A) AI use by treatment (yes/no)
# =========================================================
ai_treat_table, n_ai_included = build_binary_treat_table(
    data=d_treat,
    outcome_col="ai_use_bin",
    user_label="ai_user",
    non_user_label="non_ai_user"
)

print("\nAI use by treatment (yes/no):")
print(f"Included N for AI outcome: {n_ai_included:,}")
print(ai_treat_table)

# =========================================================
# B) Medical AI use by treatment (among AI users only)
# Same coding logic as your regression notebook
# =========================================================
purpose_cols_13 = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
purpose_cols_13 = [c for c in purpose_cols_13 if c in d_treat.columns]

non_medical_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_med_set = {x.strip().lower() for x in non_medical_answers}

def norm_str(x):
    if pd.isna(x):
        return ""
    v = str(x).strip()
    if v.lower() == "nan":
        return ""
    return v.lower()

d_ai = d_treat.loc[d_treat["ai_use_bin"] == 1].copy()

if purpose_cols_13:
    med_vals = []
    for _, row in d_ai[purpose_cols_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            med_vals.append(np.nan)
        elif any(v not in non_med_set for v in answered):
            med_vals.append(1)
        else:
            med_vals.append(0)

    d_ai["med_use"] = med_vals

    med_treat_table, n_med_included = build_binary_treat_table(
        data=d_ai,
        outcome_col="med_use",
        user_label="med_user",
        non_user_label="med_non_user"
    )

    print("\nMedical AI use by treatment (among AI users only):")
    print(f"Included N for medical outcome: {n_med_included:,}")
    print(med_treat_table)
else:
    print("\nMedical AI use table skipped: required medical-use columns not found.")

# =========================================================
# C) Emotional AI use by treatment (among AI users only)
# Same coding logic as your regression notebook
# =========================================================
purpose_cols_4_13 = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
]
purpose_cols_4_13 = [c for c in purpose_cols_4_13 if c in d_ai.columns]

non_use_answers = [
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
]
non_use_set = {x.strip().lower() for x in non_use_answers}

if purpose_cols_4_13:
    emo_vals = []
    for _, row in d_ai[purpose_cols_4_13].iterrows():
        vals = [norm_str(row[c]) for c in purpose_cols_4_13]
        answered = [v for v in vals if v != ""]

        if len(answered) == 0:
            emo_vals.append(np.nan)
        elif any(v not in non_use_set for v in answered):
            emo_vals.append(1)
        else:
            emo_vals.append(0)

    d_ai["emo_use"] = emo_vals

    emo_treat_table, n_emo_included = build_binary_treat_table(
        data=d_ai,
        outcome_col="emo_use",
        user_label="emo_user",
        non_user_label="emo_non_user"
    )

    print("\nEmotional AI use by treatment (among AI users only):")
    print(f"Included N for emotional outcome: {n_emo_included:,}")
    print(emo_treat_table)
else:
    print("\nEmotional AI use table skipped: required emotional-use columns not found.")


AI use by treatment (yes/no):
Included N for AI outcome: 12,917
ai_use_bin       n_ai_user  n_non_ai_user  n_total  pct_ai_user  \
treatment_group                                                   
no                    5412           6457    11869         45.6   
yes                    520            528     1048         49.6   

ai_use_bin       pct_non_ai_user  
treatment_group                   
no                          54.4  
yes                         50.4  

Medical AI use by treatment (among AI users only):
Included N for medical outcome: 5,928
med_use          n_med_user  n_med_non_user  n_total  pct_med_user  \
treatment_group                                                      
no                     2970            2438     5408          54.9   
yes                     322             198      520          61.9   

med_use          pct_med_non_user  
treatment_group                    
no                           45.1  
yes                          38.1  

Emotional 

## Stratified outcomes: side effects

### Side effects among all AI users, stratified by age band


In [9]:
import pandas as pd
import numpy as np

# =========================================================
# Age-stratified side effects (Yes/No) among AI users
# Special rule for social alienation:
#   Yes = -1, -2, -3
# =========================================================

# -----------------------------
# 1) AI users + age bands
# -----------------------------
ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
df_use_ki = df.loc[df[ki_use_col].astype(str).str.strip().str.lower() == 'ja'].copy()

if 'age_2026' not in df_use_ki.columns:
    df_use_ki['age_2026'] = 2026 - pd.to_numeric(df_use_ki['birth_year_filled'], errors='coerce')

df_use_ki['age_2026'] = pd.to_numeric(df_use_ki['age_2026'], errors='coerce')
df_use_ki = df_use_ki[df_use_ki['age_2026'].notna() & (df_use_ki['age_2026'] >= 18)].copy()

bins = [18, 28, 38, 48, 58, 68, 78, np.inf]
labels = ['18-27', '28-37', '38-47', '48-57', '58-67', '68-77', '78+']
df_use_ki['age_band'] = pd.cut(
    df_use_ki['age_2026'],
    bins=bins,
    labels=labels,
    right=False,
    include_lowest=True
)

# -----------------------------
# 2) Side-effect variables
# -----------------------------
side_effect_cols = {
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner': 'G2Q5.1 Social alienation',
    'ki_ai_dependency': 'G2Q6 AI dependency',
    'ki_decision_difficulty': 'G2Q7 Decision difficulty'
}

print("Column check:")
for c in side_effect_cols:
    print(f"  {c}: {'FOUND' if c in df_use_ki.columns else 'MISSING'}")

# Text-based "Yes" coding for dependency + decision difficulty
YES_VALUES_TEXT = {
    'trifft ein wenig zu',
    'trifft teilweise zu',
    'trifft völlig zu',
    'trifft voellig zu',   # fallback if oe instead of ö
}

# Numeric "Yes" coding for social alienation
SOCIAL_YES_NUM = {-1, -2, -3}

def sideeffect_yes_no(v, col):
    if pd.isna(v):
        return np.nan

    s = str(v).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan

    # Special coding for social alienation variable
    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        num = pd.to_numeric(s, errors='coerce')
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0  # all other non-missing values = No

    # Coding for dependency + decision difficulty
    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0  # all other non-missing values = No

def make_age_table(data, y_col, age_col='age_band', age_order=None):
    tmp = data.dropna(subset=[y_col]).copy()

    tab = (
        tmp.groupby(age_col, observed=False)[y_col]
           .agg(n_yes='sum', N='count')
           .reindex(age_order)
    )

    tab['n_yes'] = tab['n_yes'].fillna(0).astype(int)
    tab['N'] = tab['N'].fillna(0).astype(int)
    tab['n_no'] = tab['N'] - tab['n_yes']

    tab['pct_yes'] = np.where(tab['N'] > 0, (tab['n_yes'] / tab['N'] * 100).round(1), np.nan)
    tab['pct_no']  = np.where(tab['N'] > 0, (tab['n_no'] / tab['N'] * 100).round(1), np.nan)

    tab['yes_n/N_%'] = (
        tab['n_yes'].astype(str) + '/' + tab['N'].astype(str) + ' (' + tab['pct_yes'].astype(str) + '%)'
    )
    tab['no_n/N_%'] = (
        tab['n_no'].astype(str) + '/' + tab['N'].astype(str) + ' (' + tab['pct_no'].astype(str) + '%)'
    )

    return tab, len(tmp)

# -----------------------------
# 3) Run for all side effects
# -----------------------------
for col, label in side_effect_cols.items():
    if col not in df_use_ki.columns:
        print(f"\n[{label}] column not found -> skipped: {col}")
        continue

    y_col = f'{col}_yes'
    df_use_ki[y_col] = df_use_ki[col].apply(lambda v: sideeffect_yes_no(v, col))

    tab, n_included = make_age_table(
        data=df_use_ki,
        y_col=y_col,
        age_col='age_band',
        age_order=labels
    )
    n_missing = int(df_use_ki[y_col].isna().sum())

    print("\n" + "=" * 88)
    print(f"{label} by age group")
    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        print("Yes definition: value in {-1, -2, -3}")
    else:
        print("Yes definition: trifft ein wenig/teilweise/völlig zu")

    print(tab[['N', 'n_yes', 'pct_yes', 'yes_n/N_%', 'n_no', 'pct_no', 'no_n/N_%']])
    print(f"\nIncluded N (non-missing {label}): {n_included}")
    print(f"Missing {label} responses: {n_missing}")

    # Optional raw-value check
    print("Top raw values:")
    print(df_use_ki[col].astype(str).str.strip().value_counts(dropna=False).head(10))

Column check:
  ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner: FOUND
  ki_ai_dependency: FOUND
  ki_decision_difficulty: FOUND

G2Q5.1 Social alienation by age group
Yes definition: value in {-1, -2, -3}
             N  n_yes  pct_yes       yes_n/N_%  n_no  pct_no  \
age_band                                                       
18-27      462      3      0.6    3/462 (0.6%)   459    99.4   
28-37     1739      7      0.4   7/1739 (0.4%)  1732    99.6   
38-47     2737     16      0.6  16/2737 (0.6%)  2721    99.4   
48-57     3094      7      0.2   7/3094 (0.2%)  3087    99.8   
58-67     3954      7      0.2   7/3954 (0.2%)  3947    99.8   
68-77     2183      2      0.1   2/2183 (0.1%)  2181    99.9   
78+        424      1      0.2    1/424 (0.2%)   423    99.8   

                   no_n/N_%  
age_band                     
18-27       459/462 (99.4%)  
28-37     1732/1739 (99.6%)  
38-47     2721/2737 (99.4%)  
48-57     3087/3094 (99.8%)  
58-67     3947/3954 (99.8%)  


### Side effects among all AI users, stratified by gender


In [10]:
import pandas as pd
import numpy as np

# =========================================================
# Gender-stratified side effects (Yes/No) among AI users
# Include only participants with:
#   - answered AI-use question
#   - answered gender
#   - valid age (>=18)
# =========================================================

# -----------------------------
# 1) Start from full data
# -----------------------------
d = df.copy()

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'  # change if needed

if ki_use_col not in d.columns:
    raise KeyError(f"AI-use column not found: {ki_use_col}")
if gender_col not in d.columns:
    raise KeyError(f"Gender column not found: {gender_col}")

# -----------------------------
# 2) AI-use: require answered + keep AI users
# -----------------------------
ai_raw = d[ki_use_col].astype(str).str.strip().str.lower()
answered_ai = d[ki_use_col].notna() & (~ai_raw.isin(['', 'nan']))
ai_yes = ai_raw.isin(['ja', 'yes'])

# -----------------------------
# 3) Age: require valid age
# -----------------------------
if 'age_2026' not in d.columns:
    d['age_2026'] = 2026 - pd.to_numeric(d['birth_year_filled'], errors='coerce')

d['age_2026'] = pd.to_numeric(d['age_2026'], errors='coerce')
valid_age = d['age_2026'].notna() & (d['age_2026'] >= 18)

# -----------------------------
# 4) Gender: require answered + code female/male/other
# -----------------------------
g_raw = d[gender_col].astype(str).str.strip()
g = g_raw.str.lower()

answered_gender = d[gender_col].notna() & (~g.isin(['', 'nan']))

female_vals = {'weiblich', 'female', 'f', 'w', 'frau'}
male_vals   = {'männlich', 'maennlich', 'male', 'm', 'mann'}

is_female = answered_gender & g.isin(female_vals)
is_male   = answered_gender & g.isin(male_vals)
is_other  = answered_gender & (~is_female) & (~is_male)

d["gender_group"] = pd.Series(pd.NA, index=d.index, dtype="string")
d.loc[is_female, "gender_group"] = "female"
d.loc[is_male, "gender_group"] = "male"
d.loc[is_other, "gender_group"] = "other"

# require answered gender (i.e., got coded into one of the 3 groups)
valid_gender = d["gender_group"].notna()

# -----------------------------
# 5) FINAL BASE FILTER
# -----------------------------
mask_base = answered_ai & ai_yes & valid_age & valid_gender
d = d.loc[mask_base].copy()

group_order = ["female", "male", "other"]

print("Base inclusion counts:")
print(f"  N answered AI-use question: {int(answered_ai.sum())}")
print(f"  N AI users (yes):           {int((answered_ai & ai_yes).sum())}")
print(f"  N valid age (>=18):         {int(valid_age.sum())}")
print(f"  N answered gender:          {int(valid_gender.sum())}")
print(f"  FINAL N (all criteria):     {len(d)}")

# -----------------------------
# 6) Side-effect variables
# -----------------------------
side_effect_cols = {
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner': 'G2Q5.1 Social alienation',
    'ki_ai_dependency': 'G2Q6 AI dependency',
    'ki_decision_difficulty': 'G2Q7 Decision difficulty'
}

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    'trifft ein wenig zu',
    'trifft teilweise zu',
    'trifft völlig zu',
    'trifft voellig zu',
}

def sideeffect_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan

    # Social alienation: Yes = -1/-2/-3
    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        num = pd.to_numeric(s, errors='coerce')
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    # Dependency + decision difficulty
    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

def make_group_table(data, y_col, group_col='gender_group', group_order=None):
    tmp = data.dropna(subset=[y_col]).copy()

    tab = (
        tmp.groupby(group_col, observed=False)[y_col]
           .agg(n_yes='sum', N='count')
           .reindex(group_order)
    )

    tab['n_yes'] = tab['n_yes'].fillna(0).astype(int)
    tab['N'] = tab['N'].fillna(0).astype(int)
    tab['n_no'] = tab['N'] - tab['n_yes']
    tab['pct_yes'] = np.where(tab['N'] > 0, (tab['n_yes'] / tab['N'] * 100).round(1), np.nan)
    tab['pct_no']  = np.where(tab['N'] > 0, (tab['n_no'] / tab['N'] * 100).round(1), np.nan)
    tab['yes_n/N_%'] = tab['n_yes'].astype(str) + '/' + tab['N'].astype(str) + ' (' + tab['pct_yes'].astype(str) + '%)'
    tab['no_n/N_%']  = tab['n_no'].astype(str) + '/' + tab['N'].astype(str) + ' (' + tab['pct_no'].astype(str) + '%)'
    return tab, len(tmp)

# -----------------------------
# 7) Run analysis
# -----------------------------
for col, label in side_effect_cols.items():
    if col not in d.columns:
        print(f"\n[{label}] column missing -> skipped: {col}")
        continue

    y_col = f"{col}_yes"
    d[y_col] = d[col].apply(lambda v: sideeffect_yes_no(v, col))

    tab, n_included = make_group_table(d, y_col, group_col='gender_group', group_order=group_order)
    n_missing = int(d[y_col].isna().sum())

    print("\n" + "=" * 88)
    print(f"{label} by gender")
    print(tab[['N', 'n_yes', 'pct_yes', 'yes_n/N_%', 'n_no', 'pct_no', 'no_n/N_%']])
    print(f"\nIncluded N (non-missing {label}): {n_included}")
    print(f"Missing {label} responses: {n_missing}")

Base inclusion counts:
  N answered AI-use question: 29045
  N AI users (yes):           15132
  N valid age (>=18):         29444
  N answered gender:          29471
  FINAL N (all criteria):     15058

G2Q5.1 Social alienation by gender
                 N  n_yes  pct_yes       yes_n/N_%  n_no  pct_no  \
gender_group                                                       
female        8784     22      0.3  22/8784 (0.3%)  8762    99.7   
male          5763     21      0.4  21/5763 (0.4%)  5742    99.6   
other           41      0      0.0     0/41 (0.0%)    41   100.0   

                       no_n/N_%  
gender_group                     
female        8762/8784 (99.7%)  
male          5742/5763 (99.6%)  
other            41/41 (100.0%)  

Included N (non-missing G2Q5.1 Social alienation): 14588
Missing G2Q5.1 Social alienation responses: 470

G2Q6 AI dependency by gender
                 N  n_yes  pct_yes         yes_n/N_%  n_no  pct_no  \
gender_group                                

### Side effects among all AI users, stratified by daily functioning


In [11]:
import pandas as pd
import numpy as np

# =========================================================
# Side effects by self-rated functioning level (EQ5D5L_3)
# Include only participants with:
#   - answered AI-use question
#   - answered gender
#   - valid age (>=18)
# Then stratify by EQ5D5L_3 levels
# =========================================================

# -----------------------------
# 1) Start from full data
# -----------------------------
d = df.copy()

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'    # change if needed
eq_col = 'EQ5D5L_3'

for c in [ki_use_col, gender_col, eq_col]:
    if c not in d.columns:
        raise KeyError(f"Required column not found: {c}")

# -----------------------------
# 2) AI-use: require answered + keep AI users
# -----------------------------
ai_raw = d[ki_use_col].astype(str).str.strip().str.lower()
answered_ai = d[ki_use_col].notna() & (~ai_raw.isin(['', 'nan']))
ai_yes = ai_raw.isin(['ja', 'yes'])

# -----------------------------
# 3) Age: require valid age
# -----------------------------
if 'age_2026' not in d.columns:
    d['age_2026'] = 2026 - pd.to_numeric(d['birth_year_filled'], errors='coerce')

d['age_2026'] = pd.to_numeric(d['age_2026'], errors='coerce')
valid_age = d['age_2026'].notna() & (d['age_2026'] >= 18)

# -----------------------------
# 4) Gender: require answered + robust female/male/other coding
# -----------------------------
g = d[gender_col].astype(str).str.strip().str.lower()
answered_gender = d[gender_col].notna() & (~g.isin(['', 'nan']))

female_vals = {'weiblich', 'female', 'f', 'w', 'frau'}
male_vals   = {'männlich', 'maennlich', 'male', 'm', 'mann'}

is_female = answered_gender & g.isin(female_vals)
is_male   = answered_gender & g.isin(male_vals)
is_other  = answered_gender & (~is_female) & (~is_male)

d["gender_group"] = pd.Series(pd.NA, index=d.index, dtype="string")
d.loc[is_female, "gender_group"] = "female"
d.loc[is_male, "gender_group"] = "male"
d.loc[is_other, "gender_group"] = "other"

valid_gender = d["gender_group"].notna()

# -----------------------------
# 5) Functioning levels (EQ5D5L_3): require answered + map a-e
# -----------------------------
eq_raw = d[eq_col].astype(str).str.strip().str.lower()
answered_eq = d[eq_col].notna() & (~eq_raw.isin(['', 'nan']))

eq_map = {'a': 1, 'b': 2, 'c': 3, 'd': 4, 'e': 5}
d['eq5d_daily'] = eq_raw.map(eq_map)

eq_labels = {
    1: '1 (no problems)',
    2: '2 (slight)',
    3: '3 (moderate)',
    4: '4 (severe)',
    5: '5 (extreme)',
}
d['functioning_group'] = d['eq5d_daily'].map(eq_labels)
func_order = [eq_labels[i] for i in [1, 2, 3, 4, 5]]
valid_functioning = d['functioning_group'].notna()

# -----------------------------
# 6) FINAL BASE FILTER
# -----------------------------
mask_base = answered_ai & ai_yes & valid_age & valid_gender & valid_functioning
d = d.loc[mask_base].copy()

print("Base inclusion counts:")
print(f"  N answered AI-use question: {int(answered_ai.sum())}")
print(f"  N AI users (yes):           {int((answered_ai & ai_yes).sum())}")
print(f"  N valid age (>=18):         {int(valid_age.sum())}")
print(f"  N answered gender:          {int(valid_gender.sum())}")
print(f"  N valid EQ5D5L_3:           {int(valid_functioning.sum())}")
print(f"  FINAL N (all criteria):     {len(d)}")

# -----------------------------
# 7) Side-effect coding
# -----------------------------
side_effect_cols = {
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner': 'G2Q5.1 Social alienation',
    'ki_ai_dependency': 'G2Q6 AI dependency',
    'ki_decision_difficulty': 'G2Q7 Decision difficulty'
}

# Social alienation: Yes if -1, -2, -3
SOCIAL_YES_NUM = {-1, -2, -3}

# Dependency + decision difficulty: Yes if agreement responses
YES_VALUES_TEXT = {
    'trifft ein wenig zu',
    'trifft teilweise zu',
    'trifft völlig zu',
    'trifft voellig zu',
}

def sideeffect_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan

    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        num = pd.to_numeric(s, errors='coerce')
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

def make_group_table(data, y_col, group_col='functioning_group', group_order=None):
    tmp = data.dropna(subset=[y_col]).copy()

    tab = (
        tmp.groupby(group_col, observed=False)[y_col]
           .agg(n_yes='sum', N='count')
           .reindex(group_order)
    )

    tab['n_yes'] = tab['n_yes'].fillna(0).astype(int)
    tab['N'] = tab['N'].fillna(0).astype(int)
    tab['n_no'] = tab['N'] - tab['n_yes']
    tab['pct_yes'] = np.where(tab['N'] > 0, (tab['n_yes'] / tab['N'] * 100).round(1), np.nan)
    tab['pct_no']  = np.where(tab['N'] > 0, (tab['n_no'] / tab['N'] * 100).round(1), np.nan)
    tab['yes_n/N_%'] = tab['n_yes'].astype(str) + '/' + tab['N'].astype(str) + ' (' + tab['pct_yes'].astype(str) + '%)'
    tab['no_n/N_%']  = tab['n_no'].astype(str)  + '/' + tab['N'].astype(str) + ' (' + tab['pct_no'].astype(str) + '%)'
    return tab, len(tmp)

# -----------------------------
# 8) Run analysis
# -----------------------------
for col, label in side_effect_cols.items():
    if col not in d.columns:
        print(f"\n[{label}] column missing -> skipped: {col}")
        continue

    y_col = f"{col}_yes"
    d[y_col] = d[col].apply(lambda v: sideeffect_yes_no(v, col))

    tab, n_included = make_group_table(
        d, y_col=y_col, group_col='functioning_group', group_order=func_order
    )
    n_missing = int(d[y_col].isna().sum())

    print("\n" + "=" * 88)
    print(f"{label} by self-rated functioning level (EQ5D5L_3)")
    print(tab[['N', 'n_yes', 'pct_yes', 'yes_n/N_%', 'n_no', 'pct_no', 'no_n/N_%']])
    print(f"\nIncluded N (non-missing {label}): {n_included}")
    print(f"Missing {label} responses: {n_missing}")

Base inclusion counts:
  N answered AI-use question: 29045
  N AI users (yes):           15132
  N valid age (>=18):         29444
  N answered gender:          29471
  N valid EQ5D5L_3:           13158
  FINAL N (all criteria):     5960

G2Q5.1 Social alienation by self-rated functioning level (EQ5D5L_3)
                      N  n_yes  pct_yes      yes_n/N_%  n_no  pct_no  \
functioning_group                                                      
1 (no problems)    3031      6      0.2  6/3031 (0.2%)  3025    99.8   
2 (slight)         1753      3      0.2  3/1753 (0.2%)  1750    99.8   
3 (moderate)        717      2      0.3   2/717 (0.3%)   715    99.7   
4 (severe)          233      0      0.0   0/233 (0.0%)   233   100.0   
5 (extreme)          37      0      0.0    0/37 (0.0%)    37   100.0   

                            no_n/N_%  
functioning_group                     
1 (no problems)    3025/3031 (99.8%)  
2 (slight)         1750/1753 (99.8%)  
3 (moderate)         715/717 (99

### Side effects among all AI users, stratified by diagnosis status


In [12]:
import pandas as pd
import numpy as np

# =========================================================
# Lifetime psychiatric disorder vs side effects (among AI users)
# Keeps the same strict inclusion style:
#   - answered AI-use question
#   - AI-use = yes
#   - valid age (>=18)
#   - answered gender
#   - answered lifetime psychiatric disorder
# =========================================================

# -----------------------------
# 1) Start
# -----------------------------
d = df.copy()

ki_use_col = 'ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud'
gender_col = 'gender_filled'              # adjust if needed
diag_col = 'Diagnose_selbst_dich'         # adjust if needed

for c in [ki_use_col, gender_col, diag_col]:
    if c not in d.columns:
        raise KeyError(f"Required column not found: {c}")

# -----------------------------
# 2) AI-use filter
# -----------------------------
ai_raw = d[ki_use_col].astype(str).str.strip().str.lower()
answered_ai = d[ki_use_col].notna() & (~ai_raw.isin(['', 'nan']))
ai_yes = ai_raw.isin(['ja', 'yes'])

# -----------------------------
# 3) Age filter
# -----------------------------
if 'age_2026' not in d.columns:
    d['age_2026'] = 2026 - pd.to_numeric(d['birth_year_filled'], errors='coerce')

d['age_2026'] = pd.to_numeric(d['age_2026'], errors='coerce')
valid_age = d['age_2026'].notna() & (d['age_2026'] >= 18)

# -----------------------------
# 4) Gender filter (answered gender required)
# -----------------------------
g = d[gender_col].astype(str).str.strip().str.lower()
answered_gender = d[gender_col].notna() & (~g.isin(['', 'nan']))

female_vals = {'weiblich', 'female', 'f', 'w', 'frau'}
male_vals   = {'männlich', 'maennlich', 'male', 'm', 'mann'}

is_female = answered_gender & g.isin(female_vals)
is_male   = answered_gender & g.isin(male_vals)
is_other  = answered_gender & (~is_female) & (~is_male)

d["gender_group"] = pd.Series(pd.NA, index=d.index, dtype="string")
d.loc[is_female, "gender_group"] = "female"
d.loc[is_male, "gender_group"] = "male"
d.loc[is_other, "gender_group"] = "other"
valid_gender = d["gender_group"].notna()

# -----------------------------
# 5) Lifetime psychiatric disorder coding
# -----------------------------
diag_raw = d[diag_col].astype(str).str.strip().str.lower()

diag_yes_vals = {'a', 'ja', 'yes', '1', 'true'}
diag_no_vals  = {'b', 'nein', 'no', '0', 'false'}

d['diagnosis_group'] = np.where(
    diag_raw.isin(diag_yes_vals), 'yes_lifetime_disorder',
    np.where(diag_raw.isin(diag_no_vals), 'no_lifetime_disorder', pd.NA)
)

valid_diag = pd.Series(d['diagnosis_group']).notna()
diag_order = ['yes_lifetime_disorder', 'no_lifetime_disorder']

# -----------------------------
# 6) Final base filter
# -----------------------------
mask_base = answered_ai & ai_yes & valid_age & valid_gender & valid_diag
d = d.loc[mask_base].copy()

print("Base inclusion counts:")
print(f"  N answered AI-use question: {int(answered_ai.sum())}")
print(f"  N AI users (yes):           {int((answered_ai & ai_yes).sum())}")
print(f"  N valid age (>=18):         {int(valid_age.sum())}")
print(f"  N answered gender:          {int(valid_gender.sum())}")
print(f"  N valid lifetime diagnosis: {int(valid_diag.sum())}")
print(f"  FINAL N (all criteria):     {len(d)}")

# -----------------------------
# 7) Side-effect variables + coding
# -----------------------------
side_effect_cols = {
    'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner': 'G2Q5.1 Social alienation',
    'ki_ai_dependency': 'G2Q6 AI dependency',
    'ki_decision_difficulty': 'G2Q7 Decision difficulty'
}

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    'trifft ein wenig zu',
    'trifft teilweise zu',
    'trifft völlig zu',
    'trifft voellig zu',
}

def sideeffect_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == '' or s.lower() == 'nan':
        return np.nan

    # Social alienation: Yes if -1/-2/-3
    if col == 'ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner':
        num = pd.to_numeric(s, errors='coerce')
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    # Dependency + decision difficulty
    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

def make_group_table(data, y_col, group_col='diagnosis_group', group_order=None):
    tmp = data.dropna(subset=[y_col]).copy()

    tab = (
        tmp.groupby(group_col, observed=False)[y_col]
           .agg(n_yes='sum', N='count')
           .reindex(group_order)
    )

    tab['n_yes'] = tab['n_yes'].fillna(0).astype(int)
    tab['N'] = tab['N'].fillna(0).astype(int)
    tab['n_no'] = tab['N'] - tab['n_yes']
    tab['pct_yes'] = np.where(tab['N'] > 0, (tab['n_yes'] / tab['N'] * 100).round(1), np.nan)
    tab['pct_no']  = np.where(tab['N'] > 0, (tab['n_no'] / tab['N'] * 100).round(1), np.nan)
    tab['yes_n/N_%'] = tab['n_yes'].astype(str) + '/' + tab['N'].astype(str) + ' (' + tab['pct_yes'].astype(str) + '%)'
    tab['no_n/N_%']  = tab['n_no'].astype(str) + '/' + tab['N'].astype(str) + ' (' + tab['pct_no'].astype(str) + '%)'
    return tab, len(tmp)

# -----------------------------
# 8) Run analysis
# -----------------------------
for col, label in side_effect_cols.items():
    if col not in d.columns:
        print(f"\n[{label}] column missing -> skipped: {col}")
        continue

    y_col = f"{col}_yes"
    d[y_col] = d[col].apply(lambda v: sideeffect_yes_no(v, col))

    tab, n_included = make_group_table(
        d, y_col=y_col, group_col='diagnosis_group', group_order=diag_order
    )
    n_missing = int(d[y_col].isna().sum())

    print("\n" + "=" * 88)
    print(f"{label} by lifetime psychiatric disorder")
    print(tab[['N', 'n_yes', 'pct_yes', 'yes_n/N_%', 'n_no', 'pct_no', 'no_n/N_%']])
    print(f"\nIncluded N (non-missing {label}): {n_included}")
    print(f"Missing {label} responses: {n_missing}")

Base inclusion counts:
  N answered AI-use question: 29045
  N AI users (yes):           15132
  N valid age (>=18):         29444
  N answered gender:          29471
  N valid lifetime diagnosis: 12474
  FINAL N (all criteria):     5670

G2Q5.1 Social alienation by lifetime psychiatric disorder
                          N  n_yes  pct_yes      yes_n/N_%  n_no  pct_no  \
diagnosis_group                                                            
yes_lifetime_disorder  1022      5      0.5  5/1022 (0.5%)  1017    99.5   
no_lifetime_disorder   4465      6      0.1  6/4465 (0.1%)  4459    99.9   

                                no_n/N_%  
diagnosis_group                           
yes_lifetime_disorder  1017/1022 (99.5%)  
no_lifetime_disorder   4459/4465 (99.9%)  

Included N (non-missing G2Q5.1 Social alienation): 5487
Missing G2Q5.1 Social alienation responses: 183

G2Q6 AI dependency by lifetime psychiatric disorder
                          N  n_yes  pct_yes        yes_n/N_%  n_no  

### Side effects among all AI users, stratified by psychological treatment status


In [13]:
import pandas as pd
import numpy as np

# =========================================================
# Current treatment vs side effects (among AI users)
# Uses your setup: d = df_base.copy(), ai_use_bin from ki_col
# =========================================================

d = df_base.copy()

treat_raw_col = "Behandlung_dich"
if treat_raw_col not in d.columns:
    raise KeyError(f"Missing required column: {treat_raw_col}")

# ---------- 1) AI use binary ----------
ai_s = d[ki_col].astype(str).str.strip().str.lower()
d["ai_use_bin"] = np.where(
    ai_s.isin(["ja", "yes"]), 1,
    np.where(ai_s.isin(["nein", "no"]), 0, np.nan)
)

# Keep only AI users
d = d[d["ai_use_bin"] == 1].copy()

# ---------- 2) Current treatment coding ----------
t = d[treat_raw_col].astype(str).str.strip().str.lower()

# adapt if needed; this follows your prior a/b style
treat_yes_vals = {"a", "ja", "yes", "1", "true"}
treat_no_vals  = {"b", "nein", "no", "0", "false"}

d["treatment_group"] = np.where(
    t.isin(treat_yes_vals), "in_treatment_now",
    np.where(t.isin(treat_no_vals), "not_in_treatment_now", pd.NA)
)

# require answered treatment
d = d[d["treatment_group"].notna()].copy()
treat_order = ["in_treatment_now", "not_in_treatment_now"]

# ---------- 3) Side-effect columns ----------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

# social alienation: Yes if -1/-2/-3
SOCIAL_YES_NUM = {-1, -2, -3}

# dependency + decision difficulty: Yes if agreement
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def sideeffect_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

def make_group_table(data, y_col, group_col="treatment_group", group_order=None):
    tmp = data.dropna(subset=[y_col]).copy()

    tab = (
        tmp.groupby(group_col, observed=False)[y_col]
           .agg(n_yes="sum", N="count")
           .reindex(group_order)
    )

    tab["n_yes"] = tab["n_yes"].fillna(0).astype(int)
    tab["N"] = tab["N"].fillna(0).astype(int)
    tab["n_no"] = tab["N"] - tab["n_yes"]
    tab["pct_yes"] = np.where(tab["N"] > 0, (tab["n_yes"] / tab["N"] * 100).round(1), np.nan)
    tab["pct_no"]  = np.where(tab["N"] > 0, (tab["n_no"] / tab["N"] * 100).round(1), np.nan)

    tab["yes_n/N_%"] = tab["n_yes"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_yes"].astype(str) + "%)"
    tab["no_n/N_%"]  = tab["n_no"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_no"].astype(str) + "%)"
    return tab, len(tmp)

# ---------- 4) Run ----------
print(f"N AI users with valid treatment status: {len(d)}")

for col, label in side_effect_cols.items():
    if col not in d.columns:
        print(f"\n[{label}] column missing -> skipped: {col}")
        continue

    y_col = f"{col}_yes"
    d[y_col] = d[col].apply(lambda v: sideeffect_yes_no(v, col))

    tab, n_included = make_group_table(d, y_col=y_col, group_col="treatment_group", group_order=treat_order)
    n_missing = int(d[y_col].isna().sum())

    print("\n" + "=" * 88)
    print(f"{label} by current treatment status")
    print(tab[["N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"]])
    print(f"\nIncluded N (non-missing {label}): {n_included}")
    print(f"Missing {label} responses: {n_missing}")

N AI users with valid treatment status: 5932

G2Q5.1 Social alienation by current treatment status
                         N  n_yes  pct_yes      yes_n/N_%  n_no  pct_no  \
treatment_group                                                           
in_treatment_now       512      2      0.4   2/512 (0.4%)   510    99.6   
not_in_treatment_now  5231      8      0.2  8/5231 (0.2%)  5223    99.8   

                               no_n/N_%  
treatment_group                          
in_treatment_now        510/512 (99.6%)  
not_in_treatment_now  5223/5231 (99.8%)  

Included N (non-missing G2Q5.1 Social alienation): 5743
Missing G2Q5.1 Social alienation responses: 189

G2Q6 AI dependency by current treatment status
                         N  n_yes  pct_yes        yes_n/N_%  n_no  pct_no  \
treatment_group                                                             
in_treatment_now       515     48      9.3    48/515 (9.3%)   467    90.7   
not_in_treatment_now  5360    433      8.1  433/

### Side effects among health-related AI users only

Here, the analysis is restricted to AI users who report general health-related and/or mental health-related use (health-related use), then side effects are summarized across strata.


In [14]:
import pandas as pd
import numpy as np

# =========================================================
# FULL OVERVIEW:
# Side effects stratified by age, gender, diagnosis, treatment, EQ5D
# in subgroup: medical OR emotional AI users
# + completeness table for all 3 side-effect variables
# =========================================================

# -----------------------------
# 0) Input dataframe + key columns
# -----------------------------
d = df_base.copy() if "df_base" in globals() else df.copy()

ki_col_local = ki_col if "ki_col" in globals() else "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
gender_col = "gender_filled"
diag_col = "Diagnose_selbst_dich"
treat_col = "Behandlung_dich"
eq_col = "EQ5D5L_3"

if ki_col_local not in d.columns:
    raise KeyError(f"Missing AI-use column: {ki_col_local}")

# -----------------------------
# 1) AI users (answered + yes)
# -----------------------------
ai_raw = d[ki_col_local].astype(str).str.strip().str.lower()
answered_ai = d[ki_col_local].notna() & (~ai_raw.isin(["", "nan"]))
d["ai_use_bin"] = np.where(
    ai_raw.isin(["ja", "yes"]), 1,
    np.where(ai_raw.isin(["nein", "no"]), 0, np.nan)
)
d_ai = d[(answered_ai) & (d["ai_use_bin"] == 1)].copy()

# -----------------------------
# 2) Build med_use / emo_use (same logic as before)
# -----------------------------
non_use_answers = {
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
}

def norm_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() == "nan":
        return ""
    return s.lower()

def build_use_binary(vals):
    vals = [norm_str(v) for v in vals]
    answered = [v for v in vals if v != ""]
    if len(answered) == 0:
        return np.nan
    if any(v not in non_use_answers for v in answered):
        return 1
    return 0

med_cols = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
med_cols = [c for c in med_cols if c in d_ai.columns]

emo_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
]
emo_cols = [c for c in emo_cols if c in d_ai.columns]

if med_cols:
    d_ai["med_use"] = d_ai[med_cols].apply(lambda r: build_use_binary(r.values), axis=1)
else:
    d_ai["med_use"] = np.nan

if emo_cols:
    d_ai["emo_use"] = d_ai[emo_cols].apply(lambda r: build_use_binary(r.values), axis=1)
else:
    d_ai["emo_use"] = np.nan

# analysis subgroup: medical OR emotional users
d_sub = d_ai[(d_ai["med_use"] == 1) | (d_ai["emo_use"] == 1)].copy()

print("Base:")
print(f"  N answered AI + AI users: {len(d_ai)}")
print(f"  N medical AI users: {(d_ai['med_use'] == 1).sum()}")
print(f"  N emotional AI users: {(d_ai['emo_use'] == 1).sum()}")
print(f"  N med OR emo users: {len(d_sub)}")

# -----------------------------
# 3) Required base filters (answered age + gender)
# -----------------------------
# Age
if "age_2026" not in d_sub.columns:
    if "birth_year_filled" in d_sub.columns:
        d_sub["age_2026"] = 2026 - pd.to_numeric(d_sub["birth_year_filled"], errors="coerce")
    else:
        d_sub["age_2026"] = np.nan

d_sub["age_2026"] = pd.to_numeric(d_sub["age_2026"], errors="coerce")
valid_age = d_sub["age_2026"].notna() & (d_sub["age_2026"] >= 18)

# Gender
if gender_col in d_sub.columns:
    g = d_sub[gender_col].astype(str).str.strip().str.lower()
    is_missing_gender = d_sub[gender_col].isna() | g.isin(["", "nan"])

    female_vals = {"weiblich", "female", "f", "w", "frau"}
    male_vals = {"männlich", "maennlich", "male", "m", "mann"}

    is_female = (~is_missing_gender) & g.isin(female_vals)
    is_male = (~is_missing_gender) & g.isin(male_vals)
    is_other = (~is_missing_gender) & (~is_female) & (~is_male)

    d_sub["gender_group"] = pd.Series(pd.NA, index=d_sub.index, dtype="string")
    d_sub.loc[is_female, "gender_group"] = "female"
    d_sub.loc[is_male, "gender_group"] = "male"
    d_sub.loc[is_other, "gender_group"] = "other"
else:
    d_sub["gender_group"] = pd.NA

valid_gender = d_sub["gender_group"].notna()

# keep only those with answered age + gender (as requested)
d_sub = d_sub[valid_age & valid_gender].copy()

print(f"  N after requiring valid age + gender: {len(d_sub)}")

# -----------------------------
# 4) Stratifier variables
# -----------------------------
# Age bands
age_order = ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]
d_sub["age_band"] = pd.cut(
    d_sub["age_2026"],
    bins=[18, 28, 38, 48, 58, 68, 78, np.inf],
    labels=age_order,
    right=False,
    include_lowest=True
)

# Lifetime diagnosis
if diag_col in d_sub.columns:
    dx = d_sub[diag_col].astype(str).str.strip().str.lower()
    dx_yes = {"a", "ja", "yes", "1", "true"}
    dx_no = {"b", "nein", "no", "0", "false"}
    d_sub["diagnosis_group"] = np.where(
        dx.isin(dx_yes), "yes_lifetime_disorder",
        np.where(dx.isin(dx_no), "no_lifetime_disorder", pd.NA)
    )
else:
    d_sub["diagnosis_group"] = pd.NA
diag_order = ["yes_lifetime_disorder", "no_lifetime_disorder"]

# Current treatment
if treat_col in d_sub.columns:
    tr = d_sub[treat_col].astype(str).str.strip().str.lower()
    tr_yes = {"a", "ja", "yes", "1", "true"}
    tr_no = {"b", "nein", "no", "0", "false"}
    d_sub["treatment_group"] = np.where(
        tr.isin(tr_yes), "in_treatment_now",
        np.where(tr.isin(tr_no), "not_in_treatment_now", pd.NA)
    )
else:
    d_sub["treatment_group"] = pd.NA
treat_order = ["in_treatment_now", "not_in_treatment_now"]

# Daily functioning
if eq_col in d_sub.columns:
    eq_raw = d_sub[eq_col].astype(str).str.strip().str.lower()
    d_sub["eq5d_daily"] = eq_raw.map({"a": 1, "b": 2, "c": 3, "d": 4, "e": 5})
else:
    d_sub["eq5d_daily"] = np.nan

eq_labels = {
    1: "1 (no problems)",
    2: "2 (slight)",
    3: "3 (moderate)",
    4: "4 (severe)",
    5: "5 (extreme)",
}
func_order = [eq_labels[i] for i in [1, 2, 3, 4, 5]]
d_sub["functioning_group"] = d_sub["eq5d_daily"].map(eq_labels)

# -----------------------------
# 5) Side-effect coding
# -----------------------------
side_effect_cols = {
    "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner": "G2Q5.1 Social alienation",
    "ki_ai_dependency": "G2Q6 AI dependency",
    "ki_decision_difficulty": "G2Q7 Decision difficulty",
}

for c in side_effect_cols.keys():
    if c not in d_sub.columns:
        print(f"Warning: missing side-effect column: {c}")

SOCIAL_YES_NUM = {-1, -2, -3}
YES_VALUES_TEXT = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
}

def to_yes_no(v, col):
    if pd.isna(v):
        return np.nan
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return np.nan

    if col == "ki_mein_verhaeltnis_zu_meinen_freunden_ist_seit_meiner":
        num = pd.to_numeric(s, errors="coerce")
        if pd.notna(num) and int(num) in SOCIAL_YES_NUM:
            return 1
        return 0

    if s.lower() in YES_VALUES_TEXT:
        return 1
    return 0

for col in side_effect_cols:
    ycol = f"{col}_yes"
    if col in d_sub.columns:
        d_sub[ycol] = d_sub[col].apply(lambda x: to_yes_no(x, col))
    else:
        d_sub[ycol] = np.nan

# -----------------------------
# 6) Helpers
# -----------------------------
def make_strat_table(data, strat_col, strat_order, y_col):
    tmp = data.dropna(subset=[strat_col, y_col]).copy()

    tab = (
        tmp.groupby(strat_col, observed=False)[y_col]
           .agg(n_yes="sum", N="count")
           .reindex(strat_order)
    )

    tab["n_yes"] = tab["n_yes"].fillna(0).astype(int)
    tab["N"] = tab["N"].fillna(0).astype(int)
    tab["n_no"] = tab["N"] - tab["n_yes"]
    tab["pct_yes"] = np.where(tab["N"] > 0, (tab["n_yes"] / tab["N"] * 100).round(1), np.nan)
    tab["pct_no"] = np.where(tab["N"] > 0, (tab["n_no"] / tab["N"] * 100).round(1), np.nan)
    tab["yes_n/N_%"] = tab["n_yes"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_yes"].astype(str) + "%)"
    tab["no_n/N_%"] = tab["n_no"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_no"].astype(str) + "%)"
    return tab, len(tmp)

def is_missing_response(s: pd.Series) -> pd.Series:
    s_str = s.astype(str).str.strip()
    return s.isna() | s_str.eq("") | s_str.str.lower().eq("nan")

# -----------------------------
# 7) Run all stratified analyses
# -----------------------------
stratifiers = [
    ("age_band", "Age", age_order),
    ("gender_group", "Gender", ["female", "male", "other"]),
    ("diagnosis_group", "Lifetime diagnosis", diag_order),
    ("treatment_group", "Current treatment", treat_order),
    ("functioning_group", "Daily functioning (EQ5D5L_3)", func_order),
]

overview_rows = []

for strat_col, strat_label, strat_order in stratifiers:
    print("\n" + "#" * 96)
    print(f"{strat_label} stratification")
    print("#" * 96)

    for raw_col, outcome_label in side_effect_cols.items():
        y_col = f"{raw_col}_yes"

        tab, n_included = make_strat_table(d_sub, strat_col, strat_order, y_col)
        n_missing_outcome = int(d_sub[y_col].isna().sum())

        print("\n" + "-" * 96)
        print(f"{outcome_label}")
        print(tab[["N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"]])
        print(f"Included N (non-missing outcome + non-missing {strat_label}): {n_included}")
        print(f"Missing outcome responses in subgroup: {n_missing_outcome}")

        out = tab.reset_index().rename(columns={strat_col: "group"})
        out["stratifier"] = strat_label
        out["outcome"] = outcome_label
        out["N_analysis_base_med_or_emo"] = len(d_sub)
        out["N_included"] = n_included
        out["N_outcome_missing"] = n_missing_outcome

        overview_rows.append(
            out[[
                "stratifier", "group", "outcome",
                "N_analysis_base_med_or_emo", "N_included", "N_outcome_missing",
                "N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"
            ]]
        )

overview_long = pd.concat(overview_rows, ignore_index=True)
print("\nCreated: overview_long")
print(overview_long.head(20))

# -----------------------------
# 8) Completeness table (all 3 side effects present)
# -----------------------------
side_effect_raw_cols = list(side_effect_cols.keys())
present_df = pd.DataFrame(
    {c: ~is_missing_response(d_sub[c]) for c in side_effect_raw_cols},
    index=d_sub.index
)
d_sub["all_3_sideeffects_present"] = present_df.all(axis=1)

completeness_rows = []

for strat_col, strat_label, strat_order in stratifiers:
    tmp = d_sub[d_sub[strat_col].notna()].copy()

    ctab = (
        tmp.groupby(strat_col, observed=False)["all_3_sideeffects_present"]
           .agg(N_complete_all_3_sideeffects="sum", N_total="count")
           .reindex(strat_order)
    )

    ctab["N_complete_all_3_sideeffects"] = ctab["N_complete_all_3_sideeffects"].fillna(0).astype(int)
    ctab["N_total"] = ctab["N_total"].fillna(0).astype(int)
    ctab["N_missing_any_sideeffect"] = ctab["N_total"] - ctab["N_complete_all_3_sideeffects"]
    ctab["pct_complete_all_3"] = np.where(
        ctab["N_total"] > 0,
        (ctab["N_complete_all_3_sideeffects"] / ctab["N_total"] * 100).round(1),
        np.nan
    )

    print("\n" + "=" * 96)
    print(f"{strat_label}: completeness of all 3 side-effect variables")
    print(ctab[["N_total", "N_complete_all_3_sideeffects", "N_missing_any_sideeffect", "pct_complete_all_3"]])

    out = ctab.reset_index().rename(columns={strat_col: "group"})
    out["stratifier"] = strat_label
    completeness_rows.append(out)

completeness_overview = pd.concat(completeness_rows, ignore_index=True)
print("\nCreated: completeness_overview")
print(completeness_overview.head(20))

# Optional export:
# overview_long.to_csv("overview_sideeffects_med_or_emo.csv", index=False)
# completeness_overview.to_csv("completeness_sideeffects_med_or_emo.csv", index=False)
# overview_long.to_excel("overview_sideeffects_med_or_emo.xlsx", index=False)
# completeness_overview.to_excel("completeness_sideeffects_med_or_emo.xlsx", index=False)

Base:
  N answered AI + AI users: 15058
  N medical AI users: 8727
  N emotional AI users: 5097
  N med OR emo users: 9507
  N after requiring valid age + gender: 9507

################################################################################################
Age stratification
################################################################################################

------------------------------------------------------------------------------------------------
G2Q5.1 Social alienation
             N  n_yes  pct_yes       yes_n/N_%  n_no  pct_no  \
age_band                                                       
18-27      349      3      0.9    3/349 (0.9%)   346    99.1   
28-37     1282      5      0.4   5/1282 (0.4%)  1277    99.6   
38-47     1796     15      0.8  15/1796 (0.8%)  1781    99.2   
48-57     1879      6      0.3   6/1879 (0.3%)  1873    99.7   
58-67     2367      5      0.2   5/2367 (0.2%)  2362    99.8   
68-77     1383      1      0.1   1/1383 (0.1%) 

## Stratified downstream behaviors (health-related AI users)

### Self-treatment following AI use


In [15]:
import pandas as pd
import numpy as np

# =========================================================
# FULL OVERVIEW (single outcome):
# Outcome: ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2
# Subgroup: AI users with med_use == 1 OR emo_use == 1
# Stratifiers: age, gender, diagnosis, treatment, daily functioning
# =========================================================

# -----------------------------
# 0) Base data and key columns
# -----------------------------
d = df_base.copy() if "df_base" in globals() else df.copy()

ki_col_local = ki_col if "ki_col" in globals() else "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
gender_col = "gender_filled"
diag_col = "Diagnose_selbst_dich"
treat_col = "Behandlung_dich"
eq_col = "EQ5D5L_3"
outcome_col = "ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2"

for c in [ki_col_local, outcome_col]:
    if c not in d.columns:
        raise KeyError(f"Missing required column: {c}")

# -----------------------------
# 1) AI users
# -----------------------------
ai_raw = d[ki_col_local].astype(str).str.strip().str.lower()
answered_ai = d[ki_col_local].notna() & (~ai_raw.isin(["", "nan"]))
d["ai_use_bin"] = np.where(
    ai_raw.isin(["ja", "yes"]), 1,
    np.where(ai_raw.isin(["nein", "no"]), 0, np.nan)
)
d_ai = d[(answered_ai) & (d["ai_use_bin"] == 1)].copy()

# -----------------------------
# 2) Build med_use and emo_use
# -----------------------------
non_use_answers = {
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
}

def norm_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    return "" if s.lower() == "nan" else s.lower()

def build_use_binary(vals):
    vals = [norm_str(v) for v in vals]
    answered = [v for v in vals if v != ""]
    if len(answered) == 0:
        return np.nan
    if any(v not in non_use_answers for v in answered):
        return 1
    return 0

# medical use items (G1Q4.1-3)
med_cols = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
med_cols = [c for c in med_cols if c in d_ai.columns]

# emotional use items (G1Q4.4-13)
emo_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
]
emo_cols = [c for c in emo_cols if c in d_ai.columns]

d_ai["med_use"] = d_ai[med_cols].apply(lambda r: build_use_binary(r.values), axis=1) if med_cols else np.nan
d_ai["emo_use"] = d_ai[emo_cols].apply(lambda r: build_use_binary(r.values), axis=1) if emo_cols else np.nan

# IMPORTANT: include either medical OR emotional users
d_sub = d_ai[(d_ai["med_use"] == 1) | (d_ai["emo_use"] == 1)].copy()

print("Base counts:")
print(f"  N AI users: {len(d_ai)}")
print(f"  N med users: {(d_ai['med_use'] == 1).sum()}")
print(f"  N emo users: {(d_ai['emo_use'] == 1).sum()}")
print(f"  N med OR emo users (analysis base): {len(d_sub)}")

# -----------------------------
# 3) Keep only answered age + gender
# -----------------------------
if "age_2026" not in d_sub.columns and "birth_year_filled" in d_sub.columns:
    d_sub["age_2026"] = 2026 - pd.to_numeric(d_sub["birth_year_filled"], errors="coerce")
d_sub["age_2026"] = pd.to_numeric(d_sub.get("age_2026"), errors="coerce")
valid_age = d_sub["age_2026"].notna() & (d_sub["age_2026"] >= 18)

if gender_col in d_sub.columns:
    g = d_sub[gender_col].astype(str).str.strip().str.lower()
    is_missing_gender = d_sub[gender_col].isna() | g.isin(["", "nan"])

    female_vals = {"weiblich", "female", "f", "w", "frau"}
    male_vals = {"männlich", "maennlich", "male", "m", "mann"}

    is_female = (~is_missing_gender) & g.isin(female_vals)
    is_male = (~is_missing_gender) & g.isin(male_vals)
    is_other = (~is_missing_gender) & (~is_female) & (~is_male)

    d_sub["gender_group"] = pd.Series(pd.NA, index=d_sub.index, dtype="string")
    d_sub.loc[is_female, "gender_group"] = "female"
    d_sub.loc[is_male, "gender_group"] = "male"
    d_sub.loc[is_other, "gender_group"] = "other"
else:
    d_sub["gender_group"] = pd.NA

valid_gender = d_sub["gender_group"].notna()

# apply requested required filters
d_sub = d_sub[valid_age & valid_gender].copy()
print(f"  N after requiring valid age + gender: {len(d_sub)}")

# -----------------------------
# 4) Build stratifiers
# -----------------------------
age_order = ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]
d_sub["age_band"] = pd.cut(
    d_sub["age_2026"],
    bins=[18, 28, 38, 48, 58, 68, 78, np.inf],
    labels=age_order,
    right=False,
    include_lowest=True
)

# diagnosis
if diag_col in d_sub.columns:
    dx = d_sub[diag_col].astype(str).str.strip().str.lower()
    d_sub["diagnosis_group"] = np.where(
        dx.isin({"a", "ja", "yes", "1", "true"}), "yes_lifetime_disorder",
        np.where(dx.isin({"b", "nein", "no", "0", "false"}), "no_lifetime_disorder", pd.NA)
    )
else:
    d_sub["diagnosis_group"] = pd.NA
diag_order = ["yes_lifetime_disorder", "no_lifetime_disorder"]

# treatment
if treat_col in d_sub.columns:
    tr = d_sub[treat_col].astype(str).str.strip().str.lower()
    d_sub["treatment_group"] = np.where(
        tr.isin({"a", "ja", "yes", "1", "true"}), "in_treatment_now",
        np.where(tr.isin({"b", "nein", "no", "0", "false"}), "not_in_treatment_now", pd.NA)
    )
else:
    d_sub["treatment_group"] = pd.NA
treat_order = ["in_treatment_now", "not_in_treatment_now"]

# daily functioning
if eq_col in d_sub.columns:
    eq_raw = d_sub[eq_col].astype(str).str.strip().str.lower()
    d_sub["eq5d_daily"] = eq_raw.map({"a": 1, "b": 2, "c": 3, "d": 4, "e": 5})
else:
    d_sub["eq5d_daily"] = np.nan

eq_labels = {
    1: "1 (no problems)",
    2: "2 (slight)",
    3: "3 (moderate)",
    4: "4 (severe)",
    5: "5 (extreme)",
}
func_order = [eq_labels[i] for i in [1, 2, 3, 4, 5]]
d_sub["functioning_group"] = d_sub["eq5d_daily"].map(eq_labels)

# -----------------------------
# 5) Outcome coding
# -----------------------------
# Adjust this set if your answer labels differ:
OUTCOME_YES_VALUES = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
    "a", "ja", "yes", "1", "true"
}

def outcome_yes_no(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in ["", "nan"]:
        return np.nan
    if s in OUTCOME_YES_VALUES:
        return 1
    return 0  # all other non-missing values = No

d_sub["outcome_bin"] = d_sub[outcome_col].apply(outcome_yes_no)

print("\nTop raw values of outcome (coding check):")
print(d_sub[outcome_col].astype(str).str.strip().value_counts(dropna=False).head(15))

# -----------------------------
# 6) Stratified table helper
# -----------------------------
def make_strat_table(data, strat_col, strat_order, outcome_bin_col="outcome_bin"):
    tmp = data.dropna(subset=[strat_col, outcome_bin_col]).copy()

    tab = (
        tmp.groupby(strat_col, observed=False)[outcome_bin_col]
           .agg(n_yes="sum", N="count")
           .reindex(strat_order)
    )

    tab["n_yes"] = tab["n_yes"].fillna(0).astype(int)
    tab["N"] = tab["N"].fillna(0).astype(int)
    tab["n_no"] = tab["N"] - tab["n_yes"]
    tab["pct_yes"] = np.where(tab["N"] > 0, (tab["n_yes"] / tab["N"] * 100).round(1), np.nan)
    tab["pct_no"] = np.where(tab["N"] > 0, (tab["n_no"] / tab["N"] * 100).round(1), np.nan)
    tab["yes_n/N_%"] = tab["n_yes"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_yes"].astype(str) + "%)"
    tab["no_n/N_%"] = tab["n_no"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_no"].astype(str) + "%)"
    return tab, len(tmp)

# -----------------------------
# 7) Run full overview
# -----------------------------
stratifiers = [
    ("age_band", "Age", age_order),
    ("gender_group", "Gender", ["female", "male", "other"]),
    ("diagnosis_group", "Lifetime diagnosis", diag_order),
    ("treatment_group", "Current treatment", treat_order),
    ("functioning_group", "Daily functioning (EQ5D5L_3)", func_order),
]

overview_rows = []

for strat_col, strat_label, strat_order in stratifiers:
    tab, n_included = make_strat_table(d_sub, strat_col, strat_order, "outcome_bin")
    n_missing_outcome = int(d_sub["outcome_bin"].isna().sum())

    print("\n" + "=" * 96)
    print(f"{strat_label} vs {outcome_col}")
    print(tab[["N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"]])
    print(f"Included N (non-missing outcome + non-missing {strat_label}): {n_included}")
    print(f"Missing outcome responses in subgroup: {n_missing_outcome}")

    out = tab.reset_index().rename(columns={strat_col: "group"})
    out["stratifier"] = strat_label
    out["outcome"] = outcome_col
    out["N_analysis_base_med_or_emo"] = len(d_sub)
    out["N_included"] = n_included
    out["N_outcome_missing"] = n_missing_outcome

    overview_rows.append(
        out[
            [
                "stratifier", "group", "outcome",
                "N_analysis_base_med_or_emo", "N_included", "N_outcome_missing",
                "N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"
            ]
        ]
    )

overview_long = pd.concat(overview_rows, ignore_index=True)
print("\nCreated `overview_long`")
print(overview_long.head(20))

# Optional export:
# overview_long.to_csv("overview_single_outcome_med_or_emo_users.csv", index=False)
# overview_long.to_excel("overview_single_outcome_med_or_emo_users.xlsx", index=False)

Base counts:
  N AI users: 15058
  N med users: 8727
  N emo users: 5097
  N med OR emo users (analysis base): 9507
  N after requiring valid age + gender: 9507

Top raw values of outcome (coding check):
ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2
Nein    6644
Ja      2702
nan      161
Name: count, dtype: int64

Age vs ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2
             N  n_yes  pct_yes         yes_n/N_%  n_no  pct_no  \
age_band                                                         
18-27      348    101     29.0   101/348 (29.0%)   247    71.0   
28-37     1279    422     33.0  422/1279 (33.0%)   857    67.0   
38-47     1793    567     31.6  567/1793 (31.6%)  1226    68.4   
48-57     1887    572     30.3  572/1887 (30.3%)  1315    69.7   
58-67     2374    646     27.2  646/2374 (27.2%)  1728    72.8   
68-77     1391    320     23.0  320/1391 (23.0%)  1071    77.0   
78+        274     74     27.0    74/274 (27.0%)   200    73.0   

            

### Emotional reassurance following AI use


In [18]:
import pandas as pd
import numpy as np

# =========================================================
# FULL OVERVIEW (single outcome):
# Outcome: ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2
# Subgroup: AI users with med_use == 1 OR emo_use == 1
# Stratifiers: age, gender, diagnosis, treatment, daily functioning
# =========================================================

# -----------------------------
# 0) Base data and key columns
# -----------------------------
d = df_base.copy() if "df_base" in globals() else df.copy()

ki_col_local = ki_col if "ki_col" in globals() else "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
gender_col = "gender_filled"
diag_col = "Diagnose_selbst_dich"
treat_col = "Behandlung_dich"
eq_col = "EQ5D5L_3"
outcome_col = "ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_4"

for c in [ki_col_local, outcome_col]:
    if c not in d.columns:
        raise KeyError(f"Missing required column: {c}")

# -----------------------------
# 1) AI users
# -----------------------------
ai_raw = d[ki_col_local].astype(str).str.strip().str.lower()
answered_ai = d[ki_col_local].notna() & (~ai_raw.isin(["", "nan"]))
d["ai_use_bin"] = np.where(
    ai_raw.isin(["ja", "yes"]), 1,
    np.where(ai_raw.isin(["nein", "no"]), 0, np.nan)
)
d_ai = d[(answered_ai) & (d["ai_use_bin"] == 1)].copy()

# -----------------------------
# 2) Build med_use and emo_use
# -----------------------------
non_use_answers = {
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
}

def norm_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    return "" if s.lower() == "nan" else s.lower()

def build_use_binary(vals):
    vals = [norm_str(v) for v in vals]
    answered = [v for v in vals if v != ""]
    if len(answered) == 0:
        return np.nan
    if any(v not in non_use_answers for v in answered):
        return 1
    return 0

# medical use items (G1Q4.1-3)
med_cols = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
med_cols = [c for c in med_cols if c in d_ai.columns]

# emotional use items (G1Q4.4-13)
emo_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
]
emo_cols = [c for c in emo_cols if c in d_ai.columns]

d_ai["med_use"] = d_ai[med_cols].apply(lambda r: build_use_binary(r.values), axis=1) if med_cols else np.nan
d_ai["emo_use"] = d_ai[emo_cols].apply(lambda r: build_use_binary(r.values), axis=1) if emo_cols else np.nan

# IMPORTANT: include either medical OR emotional users
d_sub = d_ai[(d_ai["med_use"] == 1) | (d_ai["emo_use"] == 1)].copy()

print("Base counts:")
print(f"  N AI users: {len(d_ai)}")
print(f"  N med users: {(d_ai['med_use'] == 1).sum()}")
print(f"  N emo users: {(d_ai['emo_use'] == 1).sum()}")
print(f"  N med OR emo users (analysis base): {len(d_sub)}")

# -----------------------------
# 3) Keep only answered age + gender
# -----------------------------
if "age_2026" not in d_sub.columns and "birth_year_filled" in d_sub.columns:
    d_sub["age_2026"] = 2026 - pd.to_numeric(d_sub["birth_year_filled"], errors="coerce")
d_sub["age_2026"] = pd.to_numeric(d_sub.get("age_2026"), errors="coerce")
valid_age = d_sub["age_2026"].notna() & (d_sub["age_2026"] >= 18)

if gender_col in d_sub.columns:
    g = d_sub[gender_col].astype(str).str.strip().str.lower()
    is_missing_gender = d_sub[gender_col].isna() | g.isin(["", "nan"])

    female_vals = {"weiblich", "female", "f", "w", "frau"}
    male_vals = {"männlich", "maennlich", "male", "m", "mann"}

    is_female = (~is_missing_gender) & g.isin(female_vals)
    is_male = (~is_missing_gender) & g.isin(male_vals)
    is_other = (~is_missing_gender) & (~is_female) & (~is_male)

    d_sub["gender_group"] = pd.Series(pd.NA, index=d_sub.index, dtype="string")
    d_sub.loc[is_female, "gender_group"] = "female"
    d_sub.loc[is_male, "gender_group"] = "male"
    d_sub.loc[is_other, "gender_group"] = "other"
else:
    d_sub["gender_group"] = pd.NA

valid_gender = d_sub["gender_group"].notna()

# apply requested required filters
d_sub = d_sub[valid_age & valid_gender].copy()
print(f"  N after requiring valid age + gender: {len(d_sub)}")

# -----------------------------
# 4) Build stratifiers
# -----------------------------
age_order = ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]
d_sub["age_band"] = pd.cut(
    d_sub["age_2026"],
    bins=[18, 28, 38, 48, 58, 68, 78, np.inf],
    labels=age_order,
    right=False,
    include_lowest=True
)

# diagnosis
if diag_col in d_sub.columns:
    dx = d_sub[diag_col].astype(str).str.strip().str.lower()
    d_sub["diagnosis_group"] = np.where(
        dx.isin({"a", "ja", "yes", "1", "true"}), "yes_lifetime_disorder",
        np.where(dx.isin({"b", "nein", "no", "0", "false"}), "no_lifetime_disorder", pd.NA)
    )
else:
    d_sub["diagnosis_group"] = pd.NA
diag_order = ["yes_lifetime_disorder", "no_lifetime_disorder"]

# treatment
if treat_col in d_sub.columns:
    tr = d_sub[treat_col].astype(str).str.strip().str.lower()
    d_sub["treatment_group"] = np.where(
        tr.isin({"a", "ja", "yes", "1", "true"}), "in_treatment_now",
        np.where(tr.isin({"b", "nein", "no", "0", "false"}), "not_in_treatment_now", pd.NA)
    )
else:
    d_sub["treatment_group"] = pd.NA
treat_order = ["in_treatment_now", "not_in_treatment_now"]

# daily functioning
if eq_col in d_sub.columns:
    eq_raw = d_sub[eq_col].astype(str).str.strip().str.lower()
    d_sub["eq5d_daily"] = eq_raw.map({"a": 1, "b": 2, "c": 3, "d": 4, "e": 5})
else:
    d_sub["eq5d_daily"] = np.nan

eq_labels = {
    1: "1 (no problems)",
    2: "2 (slight)",
    3: "3 (moderate)",
    4: "4 (severe)",
    5: "5 (extreme)",
}
func_order = [eq_labels[i] for i in [1, 2, 3, 4, 5]]
d_sub["functioning_group"] = d_sub["eq5d_daily"].map(eq_labels)

# -----------------------------
# 5) Outcome coding
# -----------------------------
# Adjust this set if your answer labels differ:
OUTCOME_YES_VALUES = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
    "a", "ja", "yes", "1", "true"
}

def outcome_yes_no(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in ["", "nan"]:
        return np.nan
    if s in OUTCOME_YES_VALUES:
        return 1
    return 0  # all other non-missing values = No

d_sub["outcome_bin"] = d_sub[outcome_col].apply(outcome_yes_no)

print("\nTop raw values of outcome (coding check):")
print(d_sub[outcome_col].astype(str).str.strip().value_counts(dropna=False).head(15))

# -----------------------------
# 6) Stratified table helper
# -----------------------------
def make_strat_table(data, strat_col, strat_order, outcome_bin_col="outcome_bin"):
    tmp = data.dropna(subset=[strat_col, outcome_bin_col]).copy()

    tab = (
        tmp.groupby(strat_col, observed=False)[outcome_bin_col]
           .agg(n_yes="sum", N="count")
           .reindex(strat_order)
    )

    tab["n_yes"] = tab["n_yes"].fillna(0).astype(int)
    tab["N"] = tab["N"].fillna(0).astype(int)
    tab["n_no"] = tab["N"] - tab["n_yes"]
    tab["pct_yes"] = np.where(tab["N"] > 0, (tab["n_yes"] / tab["N"] * 100).round(1), np.nan)
    tab["pct_no"] = np.where(tab["N"] > 0, (tab["n_no"] / tab["N"] * 100).round(1), np.nan)
    tab["yes_n/N_%"] = tab["n_yes"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_yes"].astype(str) + "%)"
    tab["no_n/N_%"] = tab["n_no"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_no"].astype(str) + "%)"
    return tab, len(tmp)

# -----------------------------
# 7) Run full overview
# -----------------------------
stratifiers = [
    ("age_band", "Age", age_order),
    ("gender_group", "Gender", ["female", "male", "other"]),
    ("diagnosis_group", "Lifetime diagnosis", diag_order),
    ("treatment_group", "Current treatment", treat_order),
    ("functioning_group", "Daily functioning (EQ5D5L_3)", func_order),
]

overview_rows = []

for strat_col, strat_label, strat_order in stratifiers:
    tab, n_included = make_strat_table(d_sub, strat_col, strat_order, "outcome_bin")
    n_missing_outcome = int(d_sub["outcome_bin"].isna().sum())

    print("\n" + "=" * 96)
    print(f"{strat_label} vs {outcome_col}")
    print(tab[["N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"]])
    print(f"Included N (non-missing outcome + non-missing {strat_label}): {n_included}")
    print(f"Missing outcome responses in subgroup: {n_missing_outcome}")

    out = tab.reset_index().rename(columns={strat_col: "group"})
    out["stratifier"] = strat_label
    out["outcome"] = outcome_col
    out["N_analysis_base_med_or_emo"] = len(d_sub)
    out["N_included"] = n_included
    out["N_outcome_missing"] = n_missing_outcome

    overview_rows.append(
        out[
            [
                "stratifier", "group", "outcome",
                "N_analysis_base_med_or_emo", "N_included", "N_outcome_missing",
                "N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"
            ]
        ]
    )

overview_long = pd.concat(overview_rows, ignore_index=True)
print("\nCreated `overview_long`")
print(overview_long.head(20))

# Optional export:
# overview_long.to_csv("overview_single_outcome_med_or_emo_users.csv", index=False)
# overview_long.to_excel("overview_single_outcome_med_or_emo_users.xlsx", index=False)

Base counts:
  N AI users: 15058
  N med users: 8727
  N emo users: 5097
  N med OR emo users (analysis base): 9507
  N after requiring valid age + gender: 9507

Top raw values of outcome (coding check):
ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_4
Nein    7279
Ja      2036
nan      192
Name: count, dtype: int64

Age vs ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_4
             N  n_yes  pct_yes         yes_n/N_%  n_no  pct_no  \
age_band                                                         
18-27      347    141     40.6   141/347 (40.6%)   206    59.4   
28-37     1280    509     39.8  509/1280 (39.8%)   771    60.2   
38-47     1790    549     30.7  549/1790 (30.7%)  1241    69.3   
48-57     1877    380     20.2  380/1877 (20.2%)  1497    79.8   
58-67     2364    306     12.9  306/2364 (12.9%)  2058    87.1   
68-77     1382    131      9.5   131/1382 (9.5%)  1251    90.5   
78+        275     20      7.3     20/275 (7.3%)   255    92.7   

            

### Foregoing a doctor’s visit following AI use


In [19]:
import pandas as pd
import numpy as np

# =========================================================
# FULL OVERVIEW (single outcome):
# Outcome: ki_hat_der_austausch_mit_einem_ki_programm_fuer_gesundh_2
# Subgroup: AI users with med_use == 1 OR emo_use == 1
# Stratifiers: age, gender, diagnosis, treatment, daily functioning
# =========================================================

# -----------------------------
# 0) Base data and key columns
# -----------------------------
d = df_base.copy() if "df_base" in globals() else df.copy()

ki_col_local = ki_col if "ki_col" in globals() else "ki_nutzen_sie_ki_programme_wie_z_b_chatgpt_gemini_claud"
gender_col = "gender_filled"
diag_col = "Diagnose_selbst_dich"
treat_col = "Behandlung_dich"
eq_col = "EQ5D5L_3"
outcome_col = "ki_auf_einen_arztbesuch_verzichtet_haben"

for c in [ki_col_local, outcome_col]:
    if c not in d.columns:
        raise KeyError(f"Missing required column: {c}")

# -----------------------------
# 1) AI users
# -----------------------------
ai_raw = d[ki_col_local].astype(str).str.strip().str.lower()
answered_ai = d[ki_col_local].notna() & (~ai_raw.isin(["", "nan"]))
d["ai_use_bin"] = np.where(
    ai_raw.isin(["ja", "yes"]), 1,
    np.where(ai_raw.isin(["nein", "no"]), 0, np.nan)
)
d_ai = d[(answered_ai) & (d["ai_use_bin"] == 1)].copy()

# -----------------------------
# 2) Build med_use and emo_use
# -----------------------------
non_use_answers = {
    "hierfür nutze ich keine ki",
    "(bisher) kein bedarf, würde ki aber dafür nutzen",
}

def norm_str(x):
    if pd.isna(x):
        return ""
    s = str(x).strip()
    return "" if s.lower() == "nan" else s.lower()

def build_use_binary(vals):
    vals = [norm_str(v) for v in vals]
    answered = [v for v in vals if v != ""]
    if len(answered) == 0:
        return np.nan
    if any(v not in non_use_answers for v in answered):
        return 1
    return 0

# medical use items (G1Q4.1-3)
med_cols = [
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_2",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_3",
]
med_cols = [c for c in med_cols if c in d_ai.columns]

# emotional use items (G1Q4.4-13)
emo_cols = [
    "ki_bei_aengsten_und_sorgen",
    "ki_bei_selbstzweifeln_oder_niedergeschlagenheit",
    "ki_bei_einsamkeit_z_b_um_jemanden_zum_reden_zu_haben_we",
    "ki_bei_zwischenmenschlichen_konflikten",
    "ki_bei_unsicherheiten_bei_entscheidungen_um_meine_gedan",
    "ki_bei_trauer_oder_verlust",
    "ki_wenn_grosse_lebensveraenderungen_anstehen",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_4",
    "ki_wie_haeufig_nutzen_sie_ki_programme_fuer_die_folgend_5",
    "ki_fuer_sonstige_gesundheits_psychische_oder_emotionale",
]
emo_cols = [c for c in emo_cols if c in d_ai.columns]

d_ai["med_use"] = d_ai[med_cols].apply(lambda r: build_use_binary(r.values), axis=1) if med_cols else np.nan
d_ai["emo_use"] = d_ai[emo_cols].apply(lambda r: build_use_binary(r.values), axis=1) if emo_cols else np.nan

# IMPORTANT: include either medical OR emotional users
d_sub = d_ai[(d_ai["med_use"] == 1) | (d_ai["emo_use"] == 1)].copy()

print("Base counts:")
print(f"  N AI users: {len(d_ai)}")
print(f"  N med users: {(d_ai['med_use'] == 1).sum()}")
print(f"  N emo users: {(d_ai['emo_use'] == 1).sum()}")
print(f"  N med OR emo users (analysis base): {len(d_sub)}")

# -----------------------------
# 3) Keep only answered age + gender
# -----------------------------
if "age_2026" not in d_sub.columns and "birth_year_filled" in d_sub.columns:
    d_sub["age_2026"] = 2026 - pd.to_numeric(d_sub["birth_year_filled"], errors="coerce")
d_sub["age_2026"] = pd.to_numeric(d_sub.get("age_2026"), errors="coerce")
valid_age = d_sub["age_2026"].notna() & (d_sub["age_2026"] >= 18)

if gender_col in d_sub.columns:
    g = d_sub[gender_col].astype(str).str.strip().str.lower()
    is_missing_gender = d_sub[gender_col].isna() | g.isin(["", "nan"])

    female_vals = {"weiblich", "female", "f", "w", "frau"}
    male_vals = {"männlich", "maennlich", "male", "m", "mann"}

    is_female = (~is_missing_gender) & g.isin(female_vals)
    is_male = (~is_missing_gender) & g.isin(male_vals)
    is_other = (~is_missing_gender) & (~is_female) & (~is_male)

    d_sub["gender_group"] = pd.Series(pd.NA, index=d_sub.index, dtype="string")
    d_sub.loc[is_female, "gender_group"] = "female"
    d_sub.loc[is_male, "gender_group"] = "male"
    d_sub.loc[is_other, "gender_group"] = "other"
else:
    d_sub["gender_group"] = pd.NA

valid_gender = d_sub["gender_group"].notna()

# apply requested required filters
d_sub = d_sub[valid_age & valid_gender].copy()
print(f"  N after requiring valid age + gender: {len(d_sub)}")

# -----------------------------
# 4) Build stratifiers
# -----------------------------
age_order = ["18-27", "28-37", "38-47", "48-57", "58-67", "68-77", "78+"]
d_sub["age_band"] = pd.cut(
    d_sub["age_2026"],
    bins=[18, 28, 38, 48, 58, 68, 78, np.inf],
    labels=age_order,
    right=False,
    include_lowest=True
)

# diagnosis
if diag_col in d_sub.columns:
    dx = d_sub[diag_col].astype(str).str.strip().str.lower()
    d_sub["diagnosis_group"] = np.where(
        dx.isin({"a", "ja", "yes", "1", "true"}), "yes_lifetime_disorder",
        np.where(dx.isin({"b", "nein", "no", "0", "false"}), "no_lifetime_disorder", pd.NA)
    )
else:
    d_sub["diagnosis_group"] = pd.NA
diag_order = ["yes_lifetime_disorder", "no_lifetime_disorder"]

# treatment
if treat_col in d_sub.columns:
    tr = d_sub[treat_col].astype(str).str.strip().str.lower()
    d_sub["treatment_group"] = np.where(
        tr.isin({"a", "ja", "yes", "1", "true"}), "in_treatment_now",
        np.where(tr.isin({"b", "nein", "no", "0", "false"}), "not_in_treatment_now", pd.NA)
    )
else:
    d_sub["treatment_group"] = pd.NA
treat_order = ["in_treatment_now", "not_in_treatment_now"]

# daily functioning
if eq_col in d_sub.columns:
    eq_raw = d_sub[eq_col].astype(str).str.strip().str.lower()
    d_sub["eq5d_daily"] = eq_raw.map({"a": 1, "b": 2, "c": 3, "d": 4, "e": 5})
else:
    d_sub["eq5d_daily"] = np.nan

eq_labels = {
    1: "1 (no problems)",
    2: "2 (slight)",
    3: "3 (moderate)",
    4: "4 (severe)",
    5: "5 (extreme)",
}
func_order = [eq_labels[i] for i in [1, 2, 3, 4, 5]]
d_sub["functioning_group"] = d_sub["eq5d_daily"].map(eq_labels)

# -----------------------------
# 5) Outcome coding
# -----------------------------
# Adjust this set if your answer labels differ:
OUTCOME_YES_VALUES = {
    "trifft ein wenig zu",
    "trifft teilweise zu",
    "trifft völlig zu",
    "trifft voellig zu",
    "a", "ja", "yes", "1", "true"
}

def outcome_yes_no(v):
    if pd.isna(v):
        return np.nan
    s = str(v).strip().lower()
    if s in ["", "nan"]:
        return np.nan
    if s in OUTCOME_YES_VALUES:
        return 1
    return 0  # all other non-missing values = No

d_sub["outcome_bin"] = d_sub[outcome_col].apply(outcome_yes_no)

print("\nTop raw values of outcome (coding check):")
print(d_sub[outcome_col].astype(str).str.strip().value_counts(dropna=False).head(15))

# -----------------------------
# 6) Stratified table helper
# -----------------------------
def make_strat_table(data, strat_col, strat_order, outcome_bin_col="outcome_bin"):
    tmp = data.dropna(subset=[strat_col, outcome_bin_col]).copy()

    tab = (
        tmp.groupby(strat_col, observed=False)[outcome_bin_col]
           .agg(n_yes="sum", N="count")
           .reindex(strat_order)
    )

    tab["n_yes"] = tab["n_yes"].fillna(0).astype(int)
    tab["N"] = tab["N"].fillna(0).astype(int)
    tab["n_no"] = tab["N"] - tab["n_yes"]
    tab["pct_yes"] = np.where(tab["N"] > 0, (tab["n_yes"] / tab["N"] * 100).round(1), np.nan)
    tab["pct_no"] = np.where(tab["N"] > 0, (tab["n_no"] / tab["N"] * 100).round(1), np.nan)
    tab["yes_n/N_%"] = tab["n_yes"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_yes"].astype(str) + "%)"
    tab["no_n/N_%"] = tab["n_no"].astype(str) + "/" + tab["N"].astype(str) + " (" + tab["pct_no"].astype(str) + "%)"
    return tab, len(tmp)

# -----------------------------
# 7) Run full overview
# -----------------------------
stratifiers = [
    ("age_band", "Age", age_order),
    ("gender_group", "Gender", ["female", "male", "other"]),
    ("diagnosis_group", "Lifetime diagnosis", diag_order),
    ("treatment_group", "Current treatment", treat_order),
    ("functioning_group", "Daily functioning (EQ5D5L_3)", func_order),
]

overview_rows = []

for strat_col, strat_label, strat_order in stratifiers:
    tab, n_included = make_strat_table(d_sub, strat_col, strat_order, "outcome_bin")
    n_missing_outcome = int(d_sub["outcome_bin"].isna().sum())

    print("\n" + "=" * 96)
    print(f"{strat_label} vs {outcome_col}")
    print(tab[["N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"]])
    print(f"Included N (non-missing outcome + non-missing {strat_label}): {n_included}")
    print(f"Missing outcome responses in subgroup: {n_missing_outcome}")

    out = tab.reset_index().rename(columns={strat_col: "group"})
    out["stratifier"] = strat_label
    out["outcome"] = outcome_col
    out["N_analysis_base_med_or_emo"] = len(d_sub)
    out["N_included"] = n_included
    out["N_outcome_missing"] = n_missing_outcome

    overview_rows.append(
        out[
            [
                "stratifier", "group", "outcome",
                "N_analysis_base_med_or_emo", "N_included", "N_outcome_missing",
                "N", "n_yes", "pct_yes", "yes_n/N_%", "n_no", "pct_no", "no_n/N_%"
            ]
        ]
    )

overview_long = pd.concat(overview_rows, ignore_index=True)
print("\nCreated `overview_long`")
print(overview_long.head(20))

# Optional export:
# overview_long.to_csv("overview_single_outcome_med_or_emo_users.csv", index=False)
# overview_long.to_excel("overview_single_outcome_med_or_emo_users.xlsx", index=False)

Base counts:
  N AI users: 15058
  N med users: 8727
  N emo users: 5097
  N med OR emo users (analysis base): 9507
  N after requiring valid age + gender: 9507

Top raw values of outcome (coding check):
ki_auf_einen_arztbesuch_verzichtet_haben
Nein    8325
Ja      1022
nan      160
Name: count, dtype: int64

Age vs ki_auf_einen_arztbesuch_verzichtet_haben
             N  n_yes  pct_yes         yes_n/N_%  n_no  pct_no  \
age_band                                                         
18-27      349     30      8.6     30/349 (8.6%)   319    91.4   
28-37     1280    186     14.5  186/1280 (14.5%)  1094    85.5   
38-47     1792    245     13.7  245/1792 (13.7%)  1547    86.3   
48-57     1884    203     10.8  203/1884 (10.8%)  1681    89.2   
58-67     2376    210      8.8   210/2376 (8.8%)  2166    91.2   
68-77     1390    131      9.4   131/1390 (9.4%)  1259    90.6   
78+        276     17      6.2     17/276 (6.2%)   259    93.8   

                   no_n/N_%  
age_band        